# 03. Additional-load calculation
This notebook calculates additional electricity loads associated with household activity profiles in a domestic context.
It computes EV charging, space heating, space cooling, DHW, and cooking loads.
It also returns dwelling-occupancy profiles accounting for each household member.

## 3.1 Import libraries

In [53]:
import os
import re
import random
import pandas as pd
import datetime
import numpy as np
from typing import Dict, Tuple
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Get the start time
start_time = datetime.datetime.now()
print(f"Start time : {start_time}")

Start time : 2026-07-10 10:36:58.790657


## 3.2 Simulation parameters  
The following parameters control the additional-load calculations, including household size/type assumptions for heating.

In [54]:
# Define the number of households to simulate
num_households = 146                                 # Align with ENG_02 unless a different local-energy-system size is required

# Region(s) to consider when building households and calculating consumption
selected_region = 'Cen'                             # Choose among (N-O, N-E, Cen, Sud, Isl)

household_profiles_name = f'profiles_{selected_region}_{num_households}_households'


# ------------- OCCUPANCY parameters ------------- 
# Probability that each activity is performed at home (p_home=1 at home, p_home=0 away; probabilities can be adjusted).
activity_map = {
    1:  {'cat': 'home',  'p_home': 1.00},           # Sleeping/Resting
    2:  {'cat': 'home',  'p_home': 1.00},           # Personal care & dressing
    3:  {'cat': 'maybe', 'p_home': 0.50},           # Eating        (DEPENDS ON WORK/STUDY)
    4:  {'cat': 'home',  'p_home': 1.00},           # Washing dishes
    5:  {'cat': 'home',  'p_home': 1.00},           # House cleaning
    6:  {'cat': 'home',  'p_home': 1.00},           # Cooking
    8:  {'cat': 'away',  'p_home': 0.00},           # Activities outside home
    9:  {'cat': 'home',  'p_home': 1.00},           # Doing laundry
    10: {'cat': 'home',  'p_home': 1.00},           # Watching TV/PC
    11: {'cat': 'away',  'p_home': 0.00},           # Sports
    12: {'cat': 'maybe', 'p_home': 0.10},           # Social interactions
    13: {'cat': 'maybe', 'p_home': 0.00},           # Health-related activities
    14: {'cat': 'maybe', 'p_home': 0.20},           # Work        (REMOTE OPTION)
    15: {'cat': 'home',  'p_home': 1.00},           # Reading
    16: {'cat': 'home',  'p_home': 1.00},           # Leisure at home
    17: {'cat': 'home',  'p_home': 0.99},           # Using PC
    18: {'cat': 'home',  'p_home': 0.99},           # Listening to music
    19: {'cat': 'home',  'p_home': 1.00},           # Home maintenance
    20: {'cat': 'home',  'p_home': 1.00},           # Ironing
    21: {'cat': 'maybe', 'p_home': 0.30},           # Studying      (REMOTE OPTION)
    22: {'cat': 'home',  'p_home': 0.50},           # Phone use     (DEPENDS ON WORK/STUDY)
    23: {'cat': 'maybe', 'p_home': 0.90},           # Other
}



# ------------- COOLING parameters ------------- 
# 1) Read the weekly hourly cooling table

# Define cooling directories also used later
input_folder = 'input'    
additional_consumption_input_folder = os.path.join(input_folder, '03_additional_cons') # WARNING: this path is reused in the directories section
cooling_input_folder = os.path.join(additional_consumption_input_folder, "Cooling") # defined here because it is used to read cooling parameters

cooling_week_df = pd.read_excel(os.path.join(cooling_input_folder, "cooling_profiles_4_seasons.xlsx"), sheet_name= "EE cool cons kWh_el per capita", index_col="h")


# ------------- HEATING parameters ------------- 
# 1) Household-type specifications -> (family_size, min_apartment_area, max_apartment_area) (same dictionary used also in Module IV: check for consistency)
household_apartment_specs = {
    'single':                   (1,  30,  50),          # 1 person, 30–50 m²
    'couple_wo_children':       (2,  50,  70),          # 2 pp, 50–70 m²
    'couple_w_children_3':      (3,  70,  90),          # 3 pp, 70–90 m²
    'couple_w_children_4':      (4,  80, 120),          # 4 pp, 80–120 m²
    'couple_w_children_5':      (5, 100, 150),          # 5 pp, 100–150 m²
    'couple_w_children_6':      (6, 100, 150),          # 6 pp, 100–150 m²
    'single_parent_M_2':        (3,  70,  90),          # 2 pp, but assumed dwelling size as 3pp household type
    'single_parent_F_2':        (3,  70,  90),          # 2 pp, but assumed dwelling size as 3pp household type
}

# 2) Annual thermal-demand range per square metre (heating and cooling)
min_thermal_demand_per_m2 = 100                         # kWh_th/year/m² (min)
max_thermal_demand_per_m2 = 200                         # kWh_th/year/m² (max)

# 3) Consecutive absent timesteps required to turn the heat pump off (24 = 4 h)
min_absent_steps = 24



# -------------  COOKING parameters ------------- 
induction_ownership_probability = 1.00                  # (based on latest 2023 information, 6.7% of Italian households have induction hobs)
cold_season_induction_use_probability = 1.00            # if a household has an induction hob, it uses it every time it cooks outside summer
induction_consumption_kwh = 2.2 * 10 / 60               # average induction-hob consumption [kWh] = 2.2 [kW] * 10 [min] / 60 [min/h] = [kWh]
cooking_window_steps = 3                                # timestep window used to search for other cooking events close to an identified one
simultaneous_cooking_probability = 0.0                  # probability that multiple household members cook simultaneously (could be adjusted when considering flatmates situations, not included in the present study)




# ------------- EV CHARGING parameters ------------- 
ev_ownership_probability    = 1.00                      # EV ownership in the household
car_use_probability        = 1.00                       # probability of using the car for travel
km_per_step          = 6                                # average distance travelled in a 10-minute timestep
# initial_soc         = 0.65                            # % initial state of charge of EV battery
workplace_charging_probability = 0                      # probability of having a charging point at the workplace 

charging_efficiency           = 1                       # efficiency of the wallbox charging (can be adjusted)
min_dwell_steps      = 6                                # minimum continuous timesteps of EV plug-in at home to enable the home charging
home_charging_power_kw          = 3.0                   # kW home charger power
workplace_charging_power_kw        = 7.4                # kW workplace charger power

# battery capacity in kWh, specific consumption in kWh/km
ev_classes = {
    "segment_A":    {"descr":"city car",            "capacity": 20,  "cons_kwh_per_km": 0.15},         # type / battery capacity [kWh] / specific consumption [kWh/km]
    "segment_B":    {"descr":"compact cars",        "capacity": 50,  "cons_kwh_per_km": 0.16},
    "segment_C":    {"descr":"medium cars",         "capacity": 70,  "cons_kwh_per_km": 0.16}
    # "suv_compatti":  {"descr":"suv compatti",     "capacity": 60,  "cons_kwh_per_km": 0.16},         # compact SUV
    # "suv_medi":      {"descr":"suv medi",         "capacity": 80,  "cons_kwh_per_km": 0.18},         # medium size SUV
    # "segmento_D":    {"descr":"premium/lusso",    "capacity":110,  "cons_kwh_per_km": 0.18},         # premium-luxury EVs
}

# mapping household_type -> possible EV type ownership 
ev_classes_by_household_type = {
    "single":                       ["segment_A", "segment_B"],
    "couple_wo_children":           ["segment_B","segment_C"],
    "couple_w_children_3":          ["segment_B","segment_C"],
    "couple_w_children_4":          ["segment_C"],
    "couple_w_children_5":          ["segment_C"],
    "couple_w_children_6":          ["segment_C"]
}



# ------------- DHW parameters ------------- 
shower_ownership_probability = 1.0                  # es. 99% of households
morning_shower_use_probability = 0.1                # probability of taking a shower when the "personal care" activity occurs in the morning
afternoon_shower_use_probability = 0.4              # probability of taking a shower in the afternoon
simultaneous_shower_probability = 0.10              # simultaneity probability allowed (i.e., how probable is to have simultaneous showers between members within same household)


step_minutes = 10                                   # timestep duration [min]
max_shower_shift_steps = 72                         # maximum allowed postponement [ts] -> 72 ts = 12 h
alpha_util_boiler = 0.75                            # share of (Electric Storage Water Heater, ESWH) capacity that can be used by each single member within the household
gap_reset_cluster_ts = 12                           # after >=12 ts (=2h), duration to consider if include additional shower events within the same cluster
use_average_shower_for_shift = False                # if True, it uses average shower energy to calculate the minimum time required the ESWH to charge it back
                                                    # if False, it uses the shower energy consumed to calculate the time required to fill the ESWH back


# Dictionary DHW: household_type → (number of members, boiler capacity [kWh_th], charging power [kW])  
# >>> WARNING dictionary also defined in Module IV, need to be consistent <<<<<<<
dhw_specs = {
    'single':                   (1,  4.2,     1.5,    0.013),             # 1.35),     # ~80 L, 4.2 kWh, 1.5 kW, 1.3%/h static thermal losses,   1.35 kWh/24h (standing loss @T_hot=65°C and T_amb=20°C),
    'couple_wo_children':       (2,  5.2,     2,      0.013),             # 1.56),     # ~100 L, 5.2 kWh, 2 kW, 1.3%/h,                   1.56 kWh/24h,
    'couple_w_children_3':      (3,  10.5,    3,      0.007),             # 1.88),     # ~200 L, 10.5 kWh, 3 kW, 0.7%/h,                   1.88 kWh/24h
    'couple_w_children_4':      (4,  15.7,    4,      0.007),             # 2.6),      # ~300 L, 15-7 kWh, 4 kW, 0.7%/h,                 2.6 kWh/24h
    'couple_w_children_5':      (5,  15.7,    4,      0.007),             # 2.6),      # ~300 L, 15.7 kWh, 4 kW, 0.7%/h,                 2.6 kWh/24h
    'couple_w_children_6':      (6,  20.9,    6,      0.006),             # 3.6),      # 500 L, 20.9 kWh, 4 kW, 0.6%/h,                  3.6 kWh/24h,  --- (cfr noi 422 L in household da 4, caso ampio) stessa taglia delle households molto grandi
    'single_parent_M_2':        (3,  5.2,     2,      0.013),             # 1.56),     # 2-member single-parent household -> use the size assumed for 3-person households
    'single_parent_F_2':        (3,  5.2,     2,      0.013),             # 1.56),
}


# --- DHW demand calculation for one shower ---
"""
    The following assumptions are used to calculate
    the thermal energy associated with a single shower,
    in order to define the minimum and maximum values below.

    Assumptions:
        - average shower duration:                              6-8 min
        - shower flow rate:                                     8-10 L/min
        - water consumption per shower:                         60 L/shower
          (efficient case; otherwise up to 100 L/shower)

        - specific heat capacity of water:                      4.186 kJ/kg/K
        - final water temperature supplied to the user:         45°C
        - inlet/cold water temperature:                         10°C
          (in summer, 15°C could be assumed)
        - water temperature difference from boiler:             35°C
          (T_final - T_inlet)
        - basin water consumption:                              total daily amount
      
    Q_th = V_H2O * c_H2O * DeltaT
         = 60 L/shower * 4.184 kJ/kg/K * 35 K
         = 8790 kJ / 3600
         = 2.44 kWh/shower

    Average values:
        - Efficient shower
          (10 min × 6-6.3 L/min = 60-63 L):
          ~1.7-2.2 kWh_th
        - Standard shower
          (8-10 min × 8-10 L/min = 64-100 L):
          ~1.9-3.5 kWh_th

    Empirical values:
        - 1.4-1.9 kWh per shower, from Tiefenbach et al. (2017), Swiss Federal Office of Energy / ETH Zurich,based on readings from 700 homes in Zurich          --> https://pubdb.bfe.admin.ch/en/publication/download/8561
        - 0.9-2.5 kWh per shower, from Tiefenbach et al. (2019), ETH Zurich, based on a sample of 19,600 shower readings in hotels --> https://www.alexandria.unisg.ch/server/api/core/bitstreams/32e56360-b2c6-4c67-b81e-f08038871dff/content

          The hotel range may be wider because users are generally less attentive
          to consumption when they do not pay directly for it.
"""

min_shower_consumption_kwh_th = 1.4      # kWh for shower (from work Swiss Fed Office Energy, ETH over 700 dwellings, 2017) --> https://pubdb.bfe.admin.ch/en/publication/download/8561
max_shower_consumption_kwh_th = 1.9      # kWh for shower (from work Swiss Fed Office Energy, ETH over 700 dwellings, 2017) --> https://pubdb.bfe.admin.ch/en/publication/download/8561
dhw_boiler_efficiency = 1.00              # originally = 0.98, but assumed 1, so kept thermal demand and then conversion in Module IV # kWh_th/kWh_el  (but WARNING thermal losses not considered)
reference_shower_consumption_kwh_th = (min_shower_consumption_kwh_th + max_shower_consumption_kwh_th) / 2  # average consumption shower for conservative approach


# -------------- PLOT-PRINTING parameters --------------------

# percentiles
qs = [10,20,30,40,50,60,70,80,90]
# percentile groups (es 10%-90%, 20%-80%...)
bands = [(10,90,1),(20,80,1),(30,70,1),(40,60,1)]

# Number of households by grid archetype, urban, rural, mixed.
grid_archetype = {
    8: "rural",
    13: "rural",
    14: "rural",
    57: "mixed",
    144: "urban",
    146: "urban",
}

## 3.3 Directories

In [55]:
# General directory 
# input_folder previously defined in Sec 3.2
output_folder = 'output'


#---------------------- Paths input/output for calculation loads additional ------------------
# input
household_profiles_input_dir = os.path.join(output_folder, '02_household_activity_profiles_Xls') 
additional_consumption_input_folder = os.path.join(input_folder, '03_additional_cons')  #     WARNING defined in previous section
cooling_input_folder = os.path.join(additional_consumption_input_folder, "Cooling")     #     WARNING defined in previous section
heating_coefficients_path = os.path.join(additional_consumption_input_folder, "Heating", "Coefficients_heating.xlsx")
temperature_data_path  = os.path.join(additional_consumption_input_folder, "Heating", "climate_data_58091_2023.xlsx")
thermal_profile_cop_excel_path  = os.path.join(additional_consumption_input_folder, "Heating", "normalized_thermal_profile&COP.xlsx")

base_consumption_input_xls_folder = os.path.join(output_folder, '02_base_consumption_Xls')


aggregated_base_load_file = os.path.join(base_consumption_input_xls_folder, f'{num_households}_households_{selected_region}_ts_kWh_aggregated_households_base.xlsx')    # it returns, for each single household, its own conventional load time series
base_load_total_only_file = os.path.join(base_consumption_input_xls_folder, f'{num_households}_households_{selected_region}_ts_kWh_base_total_only.xlsx')               # it's the sum of all conventional loads over ALL households (it might be generated manually)


# output
additional_consumption_output_xls_folder = os.path.join(output_folder, '03_additional_consumption_Xls')


#---------------------- Paths for additional-consumption plots ------------------------

additional_consumption_output_pictures_folder = os.path.join(output_folder, '03_additional_consumption_Pics', f"{num_households}_households_{selected_region}_{grid_archetype[num_households]}")

# base_input_profiles_attiv_households_selezionati = r"C:\Users\Francesco\OneDrive - Politecnico di Milano\Lavoro_FM_Online\Dottorato FM\Mie elaborazioni\profiles-activity-luoghi-MC-GH\codice consumption\output consumption\Xls files\Prof att nucl"     <---------- sostituisci con household_profiles_input_dir  
# base_folder_output = r"C:\Users\Francesco\OneDrive - Politecnico di Milano\Lavoro_FM_Online\Dottorato FM\Mie elaborazioni\profiles-activity-luoghi-MC-GH\loads addizion"         <---------------------- sostituisci con additional_consumption_input_folder



# Create folders if do not exist
for folder in [input_folder, 
                 output_folder, 
                 additional_consumption_input_folder,
                 base_consumption_input_xls_folder, 
                 cooling_input_folder,
                 additional_consumption_output_xls_folder,
                 additional_consumption_output_pictures_folder]:
    os.makedirs(folder, exist_ok=True)

## 3.4 Selectable sections

In [56]:
RUN_ADDITIONAL_LOADS = True                                     # whether to calculate additional loads (EV, SC, SH, DHW, cooking)
RUN_ADDITIONAL_LOAD_PLOTS = False                               # whether to plot additional loads (section to be improved)


# -----
RUN_DAILY_ACTIVITY_CONSUMPTION_PLOT = True                      # Whether to generate the combined daily figure (upper: activity distribution; lower: energy consumption)
DAILY_ACTIVITY_CONSUMPTION_PLOT_SEASONS = "all"            # Seasons for which the combined daily plot is generated (use "all" for all season, or ["winter"])

# Day selection within each weekly seasonal profile.
# 0 = first day of the selected weekly profile, 1 = second day, ..., 6 = seventh day.
DAILY_ACTIVITY_CONSUMPTION_DAY_INDEX = 0

# Optional explicit date selection.
# If a date is provided for a season, it overrides DAILY_ACTIVITY_CONSUMPTION_DAY_INDEX.
# Example:
# DAILY_ACTIVITY_CONSUMPTION_DATE_BY_SEASON = {"winter": "2023-02-15"}
DAILY_ACTIVITY_CONSUMPTION_DATE_BY_SEASON = {}

# Figure style.
SHOW_DAILY_ACTIVITY_CONSUMPTION_TITLE = False
SAVE_DAILY_ACTIVITY_CONSUMPTION_PDF = True

### 3.4.1 Path for reading conventional loads file

In [57]:
# To add a plot stacked base loads + additional:
PLOT_STACKED_CONV_PLUS_ADDITIONAL = True

# Full file with every household member
BASE_LOAD_FULL_EXCEL_PATH = aggregated_base_load_file
# Lightweight file with only the aggregated sum series. If it does not exist, automatically generated from input file
BASE_LOAD_TOTAL_ONLY_EXCEL_PATH = base_load_total_only_file

# --- Information for base load to include in the plot ---
# Which datasheet has to be used:
BASE_LOAD_SHEET_KIND = "TOT"            # "TOT" = overall base load; "FIX" = only fixed load (eg refrigerator, standby)

# Name and colour in the plot
BASE_LOAD_LABEL = "Conventional"
BASE_LOAD_COLOR = "#B3B3B3"

# ---------------- Parameters plot stacked for paper ----------------
# If False, the title is not shown in the figure.
SHOW_STACKED_CONSUMPTION_TITLE = False

# Fixed colours between seasons and scenarios (to be equal to colours in Module IV)
STACKED_CONSUMPTION_COLOR_MAP = {
    "Conventional":      "#B3B3B3",
    "DHW":               "#0072B2",
    "Cooking":           "#E69F00",
    "Cooling":           "#009E73",
    "EV":                "#D55E00",
    "Heating":           "#7E57C2",
}

STACKED_CONSUMPTION_ORDER = [
    "Conventional",
    "DHW",
    "Cooking",
    "Cooling",
    "EV",
    "Heating",
]

## 3.5 Dictionaries

In [58]:
selectable_periods = ['winter', 'spring', 'summer', 'autumn'] # ['inverno', 'primavera', 'estate', 'autunno'] if needed Italian seasons' names

season_abbr_dict = {
    'winter': 'Win',
    'spring': 'Spr',
    'summer': 'Sum',
    'autumn': 'Aut'
}

season_english_dict = {
    'winter': 'Winter',
    'spring': 'Spring',
    'summer': 'Summer',
    'autumn': 'Autumn'
}

 

# Mapping season -> (start_date, end_date). WARNING: check consistency with Module I selected periods.
season_periods = {
    'winter': ('2023-02-15 00:00:00', '2023-02-21 23:50:00'),
    'spring': ('2023-04-19 00:00:00', '2023-04-25 23:50:00'),
    'summer': ('2023-07-12 00:00:00', '2023-07-18 23:50:00'),
    'autumn': ('2023-10-11 00:00:00', '2023-10-17 23:50:00'),
}


# Full output column -> abbreviation used in files/titles
categories_map = {
    'DHW_kWh_el_annual':                    'DHW',
    'Cooking_kWh_el_annual':                'Cooking',
    'EV_home_charging_kWh_el_annual':       'EV',
    'annual_km':                            'km',
    'Heating_kWh_th_annual':                'Heating Th',
    'Cooling_kWh_el_annual':                'Cooling'
}



# Compatibility dictionaries and helpers for ITA/ENG labels.
# The module uses English canonical names internally, but these helpers allow it
# to read legacy Italian outputs when necessary.
HOUSEHOLD_TYPE_ALIASES = {
    'single': 'single', 'membro_isolato': 'single',
    'couple_wo_child': 'couple_wo_children', 'couple_wo_children': 'couple_wo_children', 'coppia_SF': 'couple_wo_children',
    'couple_w_child_3': 'couple_w_children_3', 'couple_w_children_3': 'couple_w_children_3', 'coppia_CF_3': 'couple_w_children_3',
    'couple_w_child_4': 'couple_w_children_4', 'couple_w_children_4': 'couple_w_children_4', 'coppia_CF_4': 'couple_w_children_4',
    'couple_w_child_5': 'couple_w_children_5', 'couple_w_children_5': 'couple_w_children_5', 'coppia_CF_5': 'couple_w_children_5',
    'couple_w_child_6': 'couple_w_children_6', 'couple_w_children_6': 'couple_w_children_6', 'coppia_CF_6': 'couple_w_children_6',
    'mono_M_2': 'single_parent_M_2', 'single_parent_M_2': 'single_parent_M_2', 'mono_CF_M_2': 'single_parent_M_2',
    'mono_F_2': 'single_parent_F_2', 'single_parent_F_2': 'single_parent_F_2', 'mono_CF_F_2': 'single_parent_F_2',
}

SEASON_ALIASES = {
    'winter': 'winter', 'inverno': 'winter', 'inv': 'winter',
    'spring': 'spring', 'primavera': 'spring', 'pri': 'spring',
    'summer': 'summer', 'estate': 'summer', 'est': 'summer',
    'autumn': 'autumn', 'fall': 'autumn', 'autunno': 'autumn', 'aut': 'autumn',
}

COOLING_SEASON_COLUMN_ALIASES = {
    'winter': ['winter', 'inverno', 'Winter', 'Inverno'],
    'spring': ['spring', 'primavera', 'Spring', 'Primavera'],
    'summer': ['summer', 'estate', 'Summer', 'Estate'],
    'autumn': ['autumn', 'fall', 'autunno', 'Autumn', 'Fall', 'Autunno'],
}



# ------------------ Activity dictionaries for combined daily activity + consumption plots
# These labels are intentionally kept in Italian as internal keys because they
# match the ISTAT-based activity classification used in ENG_01. The legend uses
# the English labels through activity_english_dict.

aggregated_activities = [
    'Dormire/Riposo',
    'Igiene e vestirsi',
    'Mangiare',
    'Lavare stoviglie',
    'Pulizia casa',
    'Cucinare',
    'Spostamenti',
    'Attività fuori casa',
    'Lavare bucato',
    'Guardare TV/PC',
    'Sport',
    'Contatti sociali',
    'Salute',
    'Lavoro',
    'Lettura',
    'Tempo libero a casa',
    'Uso PC',
    'Ascoltare musica',
    'Manutenzione domestica',
    'Stiratura',
    'Studio',
    'Telefono',
    'Altro'
]

activity_english_dict = {
    'Dormire/Riposo': 'Sleeping/Resting',
    'Igiene e vestirsi': 'Personal care & dressing',
    'Mangiare': 'Eating',
    'Lavare stoviglie': 'Washing dishes',
    'Pulizia casa': 'House cleaning',
    'Cucinare': 'Cooking',
    'Spostamenti': 'Travelling/Commuting',
    'Attività fuori casa': 'Activities outside home',
    'Lavare bucato': 'Doing laundry',
    'Guardare TV/PC': 'Watching TV/PC',
    'Sport': 'Sports',
    'Contatti sociali': 'Social interactions',
    'Salute': 'Health-related activities',
    'Lavoro': 'Work',
    'Lettura': 'Reading',
    'Tempo libero a casa': 'Leisure at home',
    'Uso PC': 'Using PC',
    'Ascoltare musica': 'Listening to music',
    'Manutenzione domestica': 'Home maintenance',
    'Stiratura': 'Ironing',
    'Studio': 'Studying',
    'Telefono': 'Phone use',
    'Altro': 'Other'
}

activity_color_map = {
    # Rest / personal routines
    'Dormire/Riposo': '#1b2a6b',
    'Igiene e vestirsi': '#4c72b0',
    'Mangiare': '#86b6f6',

    # Domestic work
    'Lavare stoviglie': '#56b4e9',
    'Pulizia casa': '#b3d7e8',
    'Cucinare': '#009e73',
    'Lavare bucato': '#e69f00',
    'Stiratura': '#44aa99',
    'Manutenzione domestica': '#117733',

    # Mobility / outside home
    'Spostamenti': '#7fbc41',
    'Attività fuori casa': '#f0e442',

    # Leisure / media
    'Guardare TV/PC': '#d55e00',
    'Tempo libero a casa': '#dd3497',
    'Uso PC': '#f781bf',
    'Ascoltare musica': '#fbb4ae',
    'Lettura': '#b07cc6',
    'Telefono': '#2a9d8f',

    # Social / health / sport
    'Sport': '#cc6677',
    'Contatti sociali': '#882255',
    'Salute': '#aa4499',

    # Work / study
    'Lavoro': '#6a3d9a',
    'Studio': '#88ccee',

    # Residual class
    'Altro': '#999999'
}

# Activity-code mapping used by household activity-profile workbooks.
# The Excel file stores numerical activity codes, while the plot uses
# the ISTAT activity labels above as internal categories.
activity_code_to_label = {
    1:  'Dormire/Riposo',
    2:  'Igiene e vestirsi',
    3:  'Mangiare',
    4:  'Lavare stoviglie',
    5:  'Pulizia casa',
    6:  'Cucinare',
    7:  'Spostamenti',
    8:  'Attività fuori casa',
    9:  'Lavare bucato',
    10: 'Guardare TV/PC',
    11: 'Sport',
    12: 'Contatti sociali',
    13: 'Salute',
    14: 'Lavoro',
    15: 'Lettura',
    16: 'Tempo libero a casa',
    17: 'Uso PC',
    18: 'Ascoltare musica',
    19: 'Manutenzione domestica',
    20: 'Stiratura',
    21: 'Studio',
    22: 'Telefono',
    23: 'Altro',
}

## 3.6 Functions

### 3.6.0a F(x) for ITA/ENG dictionaries

In [59]:
# Function to return English household type label
def canonical_household_type(household_type):
    """Return the canonical English household-type label used by ENG_02."""
    household_type = str(household_type)
    return HOUSEHOLD_TYPE_ALIASES.get(household_type, household_type)


# Function to return ENG season key
def canonical_season(season):
    """Return the canonical English season key."""
    return SEASON_ALIASES.get(str(season).casefold(), str(season))


# Function to normalize columns in household profiles
def normalize_household_profile_column_name(column_name, default_season=None):
    """Normalize household-profile columns to Household_X_Member_Y_<season>_<type>."""
    if not isinstance(column_name, str) or column_name == 'Datetime':
        return column_name

    match = re.match(r'^(Household|Nucleo)_(\d+)_(Member|Membro)_(\d+)_(.+)$', column_name)
    if not match:
        return column_name

    household_number = match.group(2)
    member_number = match.group(4)
    tail = match.group(5)

    season_candidates = sorted(SEASON_ALIASES.keys(), key=len, reverse=True)
    for season_candidate in season_candidates:
        if tail == season_candidate or tail.startswith(season_candidate + '_'):
            season_key = canonical_season(season_candidate)
            raw_household_type = tail[len(season_candidate):].lstrip('_')
            household_type = canonical_household_type(raw_household_type)
            return f'Household_{household_number}_Member_{member_number}_{season_key}_{household_type}'

    if default_season is not None:
        season_key = canonical_season(default_season)
        household_type = canonical_household_type(tail)
        return f'Household_{household_number}_Member_{member_number}_{season_key}_{household_type}'

    return column_name


# Function to normalize household profiles in dataset
def normalize_household_profile_columns(df, default_season=None):
    """Normalize all household-profile columns in a DataFrame."""
    df = df.copy()
    df.columns = [normalize_household_profile_column_name(c, default_season) for c in df.columns]
    return df


# Function to find column using english keys
def find_compatible_column(df, canonical_key, aliases_dict):
    """Find a dataframe column using English canonical keys plus legacy aliases."""
    aliases = aliases_dict.get(canonical_key, [canonical_key])
    normalized = {re.sub(r'[^a-z0-9]+', '', str(c).casefold()): c for c in df.columns}
    for alias in aliases:
        key = re.sub(r'[^a-z0-9]+', '', str(alias).casefold())
        if key in normalized:
            return normalized[key]
    raise KeyError(f"No compatible column found for '{canonical_key}'. Available columns: {list(df.columns)}")

### 3.6.0 F(x) defining base loads for the overall plot

In [60]:
# Function to normalize Excel sheet names
def _normalize_excel_sheet_name(name):
    """
    Normalize Excel sheet names:
      - ignore case
      - collapse multiple spaces
      - strip leading/trailing spaces
    """
    s = str(name).strip()
    s = re.sub(r"\s+", " ", s)
    return s.casefold()


# Function to find the conventional-load workbook sheets robustly
def _find_base_load_sheet_name(base_load_xls, season, sheet_kind="TOT"):
    """
    Find the base-load sheet robustly with respect to:
      - Italian/English labels
      - case
      - repeated spaces

    Accepted examples:
      inverno TOT
      Winter TOT
      winter TOT
      WINTER TOT
    """
    season = str(season).strip().lower()
    sheet_kind = str(sheet_kind).strip()

    season_aliases = {
        "winter": ["winter", "inverno"],
        "spring": ["spring", "primavera"],
        "summer": ["summer", "estate"],
        "autumn": ["autumn", "fall", "autunno"],
    }

    aliases = season_aliases.get(season, [season])

    candidate_names = []
    for alias in aliases:
        candidate_names.append(f"{alias} {sheet_kind}")
        candidate_names.append(f"{alias.capitalize()} {sheet_kind}")
        candidate_names.append(f"{alias.upper()} {sheet_kind}")

    available_norm = {
        _normalize_excel_sheet_name(sheet): sheet
        for sheet in base_load_xls.sheet_names
    }

    for candidate in candidate_names:
        candidate_norm = _normalize_excel_sheet_name(candidate)
        if candidate_norm in available_norm:
            return available_norm[candidate_norm]

    print(
        f"[WARN][BASE LOAD] No sheet found for season='{season}', "
        f"sheet_kind='{sheet_kind}'."
    )
    print(f"[WARN][BASE LOAD] Tried names: {candidate_names}")
    print(f"[WARN][BASE LOAD] Available sheets: {base_load_xls.sheet_names}")

    return None


# Function to keep only numeric columns
def _extract_numeric_base_load_df(df_base):
    """
    Keep only numeric columns and discard textual columns/metadata.
    """
    df_base_num = df_base.apply(pd.to_numeric, errors="coerce").fillna(0.0)

    if df_base_num.shape[1] == 0:
        return None

    return df_base_num


# Function to automatically create the aggregated file if missing
def _build_base_load_total_only_excel_if_missing(
    full_xlsx,
    total_only_xlsx,
    sheet_kind="TOT",
    value_col=None,
):
    """
    Automatically create the lightweight Excel file of aggregated base loads
    if it does not already exist.

    Logic:
      1) if the aggregated file exists, use it;
      2) if it does not exist, try to create it from the full file;
      3) if the full file is also missing, return None.
    """
    if value_col is None:
        value_col = BASE_LOAD_LABEL

    # If the lightweight file already exists, do not recreate it.
    if os.path.exists(total_only_xlsx):
        print(f"[INFO][BASE LOAD] Using existing aggregated file: {total_only_xlsx}")
        return total_only_xlsx

    # If the full file is missing, the aggregated one cannot be created.
    if not os.path.exists(full_xlsx):
        print(f"[WARN][BASE LOAD] Aggregated file not found: {total_only_xlsx}")
        print(f"[WARN][BASE LOAD] Full file not found: {full_xlsx}")
        print("[WARN][BASE LOAD] The stacked conventional+additional plot will be skipped.")
        return None

    print("[INFO][BASE LOAD] Aggregated file not found.")
    print(f"[INFO][BASE LOAD] Automatically creating: {total_only_xlsx}")
    print(f"[INFO][BASE LOAD] Full source file: {full_xlsx}")

    xls = pd.ExcelFile(full_xlsx)
    os.makedirs(os.path.dirname(total_only_xlsx), exist_ok=True)

    created_any_sheet = False

    with pd.ExcelWriter(total_only_xlsx, engine="openpyxl") as writer:
        for season in selectable_periods:
            sheet_name = _find_base_load_sheet_name(
                base_load_xls=xls,
                season=season,
                sheet_kind=sheet_kind,
            )

            if sheet_name is None:
                print(f"[WARN][BASE LOAD] Skipping '{season}': base-load sheet not found.")
                continue

            df_base = pd.read_excel(
                xls,
                sheet_name=sheet_name,
                index_col=0,
            )

            if df_base.empty:
                print(f"[WARN][BASE LOAD] Skipping '{season}': sheet '{sheet_name}' is empty.")
                continue

            # Try to preserve the DatetimeIndex.
            idx_dt = pd.to_datetime(df_base.index, errors="coerce")
            if idx_dt.notna().sum() == len(idx_dt):
                df_base.index = idx_dt

            df_base_num = _extract_numeric_base_load_df(df_base)

            if df_base_num is None:
                print(
                    f"[WARN][BASE LOAD] Skipping '{season}': no numeric columns "
                    f"found in sheet '{sheet_name}'."
                )
                continue

            # This calculation can be computationally expensive, but it is only
            # performed when the aggregated file does not already exist.
            base_total = df_base_num.sum(axis=1).rename(value_col)

            df_out = pd.DataFrame({value_col: base_total})

            # Save the profile sheets using the same naming convention as the notebook.
            out_sheet = f"{season} {sheet_kind}"
            df_out.to_excel(writer, sheet_name=out_sheet)
            created_any_sheet = True

            print(
                f"[OK][BASE LOAD] Aggregated '{sheet_name}' -> '{out_sheet}' "
                f"({len(df_out)} time steps)"
            )

    if not created_any_sheet:
        print("[WARN][BASE LOAD] No aggregated sheet was created.")
        if os.path.exists(total_only_xlsx):
            os.remove(total_only_xlsx)
        return None

    print(f"[OK][BASE LOAD] Saved aggregated file: {total_only_xlsx}")
    return total_only_xlsx


# Function returning the aggregated conventional-load file
def _resolve_base_load_excel_path():
    """
    Return the Excel file to use for the base+additional plot.

    Automatic logic:
      1) look for the already aggregated lightweight file;
      2) if it does not exist, create it from the full file;
      3) if the full file does not exist either, skip the plot.
    """
    if not PLOT_STACKED_CONV_PLUS_ADDITIONAL:
        return None

    return _build_base_load_total_only_excel_if_missing(
        full_xlsx=BASE_LOAD_FULL_EXCEL_PATH,
        total_only_xlsx=BASE_LOAD_TOTAL_ONLY_EXCEL_PATH,
        sheet_kind=BASE_LOAD_SHEET_KIND,
        value_col=BASE_LOAD_LABEL,
    )


# Function to read the conventional-load file
def _read_base_load_total_for_season(
    base_load_xls,
    season,
    target_index,
    sheet_kind="TOT",
):
    """
    Read the aggregated profile for one season from the lightweight Excel file.

    Accept sheet names both in Italian and English:
      - inverno TOT / Winter TOT / winter TOT
      - primavera TOT / Spring TOT / spring TOT
      - estate TOT / Summer TOT / summer TOT
      - autunno TOT / Autumn TOT / autumn TOT

    The expected file is the already aggregated one, with a single column:
      Conventional
    """
    sheet_name = _find_base_load_sheet_name(
        base_load_xls=base_load_xls,
        season=season,
        sheet_kind=sheet_kind,
    )

    if sheet_name is None:
        return None

    df_base = pd.read_excel(
        base_load_xls,
        sheet_name=sheet_name,
        index_col=0,
    )

    if df_base.empty:
        print(f"[WARN][BASE LOAD] Sheet '{sheet_name}' is empty.")
        return None

    idx_dt = pd.to_datetime(df_base.index, errors="coerce")
    if idx_dt.notna().sum() == len(idx_dt):
        df_base.index = idx_dt

    df_base_num = _extract_numeric_base_load_df(df_base)

    if df_base_num is None:
        print(f"[WARN][BASE LOAD] No numeric columns found in sheet '{sheet_name}'.")
        return None

    # Ideal case: lightweight file already aggregated with a Conventional column.
    if BASE_LOAD_LABEL in df_base_num.columns:
        base_total = df_base_num[BASE_LOAD_LABEL].copy()
        base_total.name = BASE_LOAD_LABEL

    # Robust fallback: if for some reason a full file is read, sum all numeric columns.
    else:
        print(
            f"[WARN][BASE LOAD] Column '{BASE_LOAD_LABEL}' not found in sheet "
            f"'{sheet_name}'. Using fallback: sum of all numeric columns."
        )
        base_total = df_base_num.sum(axis=1)
        base_total.name = BASE_LOAD_LABEL

    # Align the base-load index with the additional-load index.
    if isinstance(target_index, pd.DatetimeIndex) and isinstance(base_total.index, pd.DatetimeIndex):
        base_total_reindexed = base_total.reindex(target_index)

        if base_total_reindexed.isna().mean() > 0.5 and len(base_total) == len(target_index):
            base_total = pd.Series(
                base_total.to_numpy(),
                index=target_index,
                name=BASE_LOAD_LABEL,
            )
        else:
            base_total = base_total_reindexed

    elif len(base_total) == len(target_index):
        base_total = pd.Series(
            base_total.to_numpy(),
            index=target_index,
            name=BASE_LOAD_LABEL,
        )

    else:
        print(
            f"[WARN][BASE LOAD] Cannot align the base load for '{season}'. "
            f"Base length={len(base_total)}, additional length={len(target_index)}."
        )
        return None

    base_total = pd.to_numeric(base_total, errors="coerce").fillna(0.0)

    return base_total


# Function to format dates printed on consumption plots
def _format_day_month_en(x, pos=None):
    """
    Format x-axis dates as DD-Mmm without year, e.g. 15-Feb.
    Does not depend on the system locale.
    """
    month_abbr = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
    ]

    dt = mdates.num2date(x)
    return f"{dt.day:02d}-{month_abbr[dt.month - 1]}"


# Function to print additional+conventional consumption plots
def plot_stacked_consumption_for_paper(
        df_plot,
        title,
        file_name,
        output_folder,
        column_order=None,
        color_map=None,
        show_title=False,
        fig_size=(7.20, 3.70),
        font_size=9.5,
        label_font_size=10.5,
        tick_font_size=9,
        legend_font_size=8,
        title_font_size=11,
        legend_position="right",
        legend_ncol=1,
        y_label="Energy demand [kWh]",
        alpha=0.95,
        save_pdf=True):
    """
    Journal-ready stacked-area plot for conventional + additional electricity demand.

    Main options:
    - show_title=False removes the title from the figure.
    - y_label="Energy demand [kWh]" sets the vertical axis label.
    - x-axis dates are shown as DD-Mmm, without year.
    - save_pdf=True saves both PNG and PDF.
    """

    if column_order is None:
        column_order = list(df_plot.columns)

    if color_map is None:
        color_map = {}

    # Keep only available columns, preserving the desired order.
    columns_to_plot = [
        col for col in column_order
        if col in df_plot.columns
    ]

    # Add any unexpected columns at the end, instead of silently dropping them.
    columns_to_plot += [
        col for col in df_plot.columns
        if col not in columns_to_plot
    ]

    plot_df = (
        df_plot[columns_to_plot]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .clip(lower=0.0)
    )

    colors = [
        color_map.get(col, "#999999")
        for col in plot_df.columns
    ]

    with plt.rc_context({
        "font.size": font_size,
        "axes.labelsize": label_font_size,
        "xtick.labelsize": tick_font_size,
        "ytick.labelsize": tick_font_size,
        "legend.fontsize": legend_font_size,
        "axes.titlesize": title_font_size,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none"
    }):

        fig, ax = plt.subplots(figsize=fig_size)

        if isinstance(plot_df.index, pd.DatetimeIndex):
            x = plot_df.index

            ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
            ax.xaxis.set_major_formatter(plt.FuncFormatter(_format_day_month_en))
            ax.set_xlim(plot_df.index.min(), plot_df.index.max())

        else:
            x = np.arange(len(plot_df))

        ax.stackplot(
            x,
            [plot_df[col].to_numpy(dtype=float) for col in plot_df.columns],
            labels=plot_df.columns,
            colors=colors,
            alpha=alpha,
            linewidth=0.15,
            edgecolor="white"
        )

        ax.set_ylabel(y_label)
        ax.set_xlabel("")

        if show_title:
            ax.set_title(title, pad=8)

        ax.grid(axis="y", linestyle="-", linewidth=0.4, alpha=0.35)
        ax.grid(axis="x", visible=False)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.tick_params(axis="x", rotation=0)
        ax.margins(x=0)

        if legend_position == "right":
            ax.legend(
                loc="upper left",
                bbox_to_anchor=(1.01, 1.00),
                frameon=False,
                handlelength=1.2,
                borderaxespad=0.0
            )

        elif legend_position == "bottom":
            ax.legend(
                loc="upper center",
                bbox_to_anchor=(0.5, -0.18),
                ncol=legend_ncol,
                frameon=False,
                handlelength=1.2,
                columnspacing=1.0,
                labelspacing=0.6
            )

        else:
            ax.legend(
                loc=legend_position,
                frameon=True,
                framealpha=0.95
            )

        os.makedirs(output_folder, exist_ok=True)

        output_path = os.path.join(output_folder, file_name)

        fig.savefig(
            output_path,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.04
        )

        if save_pdf:
            pdf_output_path = os.path.splitext(output_path)[0] + ".pdf"
            fig.savefig(
                pdf_output_path,
                bbox_inches="tight",
                pad_inches=0.04
            )

        plt.close(fig)

### 3.6.0b Functions for combined daily activity + consumption plot

In [61]:
def _season_selected_for_daily_plot(season, selected_seasons):
    """
    Return True if the combined daily plot should be generated for this season.
    """
    if selected_seasons is None:
        return False

    if selected_seasons == "all":
        return True

    if isinstance(selected_seasons, str):
        return season == selected_seasons

    return season in selected_seasons


def _ensure_datetime_index(df, datetime_col="Datetime"):
    """
    Return a copy of df with a DatetimeIndex.

    If df already has a DatetimeIndex, it is returned with a sorted index.
    If a datetime_col exists, it is converted to datetime and set as index.
    """
    out = df.copy()

    if isinstance(out.index, pd.DatetimeIndex):
        out = out.sort_index()
        return out

    if datetime_col not in out.columns:
        raise KeyError(
            f"Cannot create a DatetimeIndex: column '{datetime_col}' not found."
        )

    out[datetime_col] = pd.to_datetime(out[datetime_col])
    out = out.set_index(datetime_col).sort_index()

    return out


def _extract_one_day(
    df,
    selected_date=None,
    day_index=0,
    expected_steps=144,
    label="dataframe"
):
    """
    Extract one complete day from a DatetimeIndex DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame with DatetimeIndex, or with a Datetime column already
        converted before calling this function.
    selected_date : str or pandas-compatible datetime, optional
        Explicit day to extract, e.g. "2023-02-15".
        If None, day_index is used.
    day_index : int
        Position of the day within the available weekly profile.
        0 means first day.
    expected_steps : int
        Expected number of 10-minute timesteps in one day.
    label : str
        Label used only in warning messages.

    Returns
    -------
    tuple[pd.DataFrame, pd.Timestamp]
        Daily DataFrame and selected day as normalized Timestamp.
    """
    df = _ensure_datetime_index(df)

    available_days = pd.Index(df.index.normalize().unique()).sort_values()

    if len(available_days) == 0:
        raise ValueError(f"No days found in {label}.")

    if selected_date is None:
        if day_index < 0 or day_index >= len(available_days):
            raise IndexError(
                f"day_index={day_index} is out of range for {label}. "
                f"Available days: {list(available_days)}"
            )

        selected_day = pd.Timestamp(available_days[day_index]).normalize()

    else:
        selected_day = pd.Timestamp(selected_date).normalize()

        if selected_day not in available_days:
            raise ValueError(
                f"Selected date {selected_day.date()} not found in {label}. "
                f"Available days: {[pd.Timestamp(d).date() for d in available_days]}"
            )

    daily_df = df.loc[df.index.normalize() == selected_day].copy()

    if len(daily_df) != expected_steps:
        print(
            f"[WARN][DAILY PLOT] {label}: selected day {selected_day.date()} "
            f"has {len(daily_df)} timesteps instead of {expected_steps}."
        )

    return daily_df, selected_day


def build_daily_activity_distribution(
    activity_day_df,
    activity_code_to_label,
    categories
):
    """
    Convert a person-level activity-code DataFrame into an aggregated activity
    distribution for one day.

    Input:
        rows    = 10-minute timesteps
        columns = household members
        values  = numerical activity codes

    Output:
        rows    = same 10-minute timesteps
        columns = activity categories
        values  = number of household members performing each activity
    """
    activity_values = activity_day_df.copy()

    # Remove a possible Datetime column if the dataframe was not indexed yet.
    activity_values = activity_values.drop(columns=["Datetime"], errors="ignore")

    # Keep only numerical activity-code values.
    activity_values = activity_values.apply(pd.to_numeric, errors="coerce")

    activity_distribution_df = pd.DataFrame(index=activity_values.index)

    for activity_code, activity_label in activity_code_to_label.items():
        activity_distribution_df[activity_label] = (
            activity_values.eq(activity_code).sum(axis=1).astype(float)
        )

    activity_distribution_df = (
        activity_distribution_df
        .reindex(columns=categories, fill_value=0.0)
        .fillna(0.0)
    )

    return activity_distribution_df


def _format_hour_axis(ax, x_index, tick_interval_hours=2):
    """
    Format a shared x-axis as hour of the day.
    """
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=tick_interval_hours))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    if len(x_index) > 0:
        ax.set_xlim(x_index.min(), x_index.max())


def plot_daily_activity_and_consumption(
        activity_week_df,
        consumption_week_df,
        title,
        file_name,
        output_folder,
        selected_date=None,
        day_index=0,
        activity_categories=None,
        activity_code_to_label=None,
        activity_english_dict=None,
        activity_color_map=None,
        consumption_column_order=None,
        consumption_color_map=None,
        expected_steps_per_day=144,
        show_title=False,
        fig_size=(7.20, 5.80),
        font_size=9.0,
        label_font_size=10.0,
        tick_font_size=8.5,
        activity_legend_font_size=6.2,
        consumption_legend_font_size=8.0,
        title_font_size=11.0,
        activity_alpha=0.90,
        consumption_alpha=0.95,
        save_pdf=True):
    """
    Create a two-panel daily plot with a shared x-axis.

    Upper panel:
        aggregated activity distribution across household members.

    Lower panel:
        stacked conventional + additional energy demand.

    The function does not replace plot_stacked_consumption_for_paper; it is an
    optional additional plot for comparing activities and loads on the same day.
    """
    if activity_categories is None:
        activity_categories = aggregated_activities

    if activity_code_to_label is None:
        activity_code_to_label = globals()["activity_code_to_label"]

    if activity_english_dict is None:
        activity_english_dict = globals()["activity_english_dict"]

    if activity_color_map is None:
        activity_color_map = globals()["activity_color_map"]

    if consumption_column_order is None:
        consumption_column_order = list(consumption_week_df.columns)

    if consumption_color_map is None:
        consumption_color_map = {}

    # 1) Extract the same selected day from activity and consumption profiles.
    activity_day_df, selected_day = _extract_one_day(
        activity_week_df,
        selected_date=selected_date,
        day_index=day_index,
        expected_steps=expected_steps_per_day,
        label="activity profile"
    )

    consumption_day_df, _ = _extract_one_day(
        consumption_week_df,
        selected_date=selected_day,
        day_index=0,
        expected_steps=expected_steps_per_day,
        label="consumption profile"
    )

    # 2) Build the upper-panel activity distribution.
    activity_distribution_df = build_daily_activity_distribution(
        activity_day_df=activity_day_df,
        activity_code_to_label=activity_code_to_label,
        categories=activity_categories
    )

    # 3) Prepare the lower-panel consumption dataframe.
    columns_to_plot = [
        col for col in consumption_column_order
        if col in consumption_day_df.columns
    ]

    columns_to_plot += [
        col for col in consumption_day_df.columns
        if col not in columns_to_plot
    ]

    consumption_day_df = (
        consumption_day_df[columns_to_plot]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .clip(lower=0.0)
    )

    # If indexes do not match but lengths do, align by position.
    # This is useful when one input was read from Excel and lost exact datetime metadata.
    if not consumption_day_df.index.equals(activity_distribution_df.index):
        if len(consumption_day_df) == len(activity_distribution_df):
            consumption_day_df.index = activity_distribution_df.index
        else:
            consumption_day_df = (
                consumption_day_df
                .reindex(activity_distribution_df.index)
                .fillna(0.0)
            )

    x = activity_distribution_df.index

    with plt.rc_context({
        "font.size": font_size,
        "axes.labelsize": label_font_size,
        "xtick.labelsize": tick_font_size,
        "ytick.labelsize": tick_font_size,
        "axes.titlesize": title_font_size,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none"
    }):

        fig, (ax_activity, ax_consumption) = plt.subplots(
            nrows=2,
            ncols=1,
            figsize=fig_size,
            sharex=True,
            gridspec_kw={
                "height_ratios": [1.05, 1.00],
                "hspace": 0.08
            }
        )

        # -----------------------------
        # Upper panel: activity profiles
        # -----------------------------
        activity_bottom = np.zeros(len(activity_distribution_df), dtype=float)

        for activity_label in activity_categories:
            if activity_label not in activity_distribution_df.columns:
                continue

            values = activity_distribution_df[activity_label].to_numpy(dtype=float)

            if np.isclose(values.sum(), 0.0):
                continue

            color = activity_color_map.get(activity_label, "#999999")
            legend_label = activity_english_dict.get(activity_label, activity_label)

            ax_activity.fill_between(
                x,
                activity_bottom,
                activity_bottom + values,
                label=legend_label,
                color=color,
                alpha=activity_alpha,
                linewidth=0.15,
                edgecolor="white"
            )

            activity_bottom = activity_bottom + values

        ax_activity.set_ylabel("Number of people")

        if len(activity_bottom) > 0 and activity_bottom.max() > 0:
            ax_activity.set_ylim(0, activity_bottom.max() * 1.03)

        ax_activity.grid(axis="y", linestyle="-", linewidth=0.4, alpha=0.40)
        ax_activity.grid(axis="x", visible=False)
        ax_activity.spines["top"].set_visible(False)
        ax_activity.spines["right"].set_visible(False)

        activity_handles, activity_labels = ax_activity.get_legend_handles_labels()
        ax_activity.legend(
            activity_handles,
            activity_labels,
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            frameon=False,
            fontsize=activity_legend_font_size,
            handlelength=1.2,
            borderaxespad=0.0
        )

        # ---------------------------------------
        # Lower panel: conventional + additional
        # ---------------------------------------
        consumption_colors = [
            consumption_color_map.get(col, "#999999")
            for col in consumption_day_df.columns
        ]

        ax_consumption.stackplot(
            x,
            [consumption_day_df[col].to_numpy(dtype=float) for col in consumption_day_df.columns],
            labels=consumption_day_df.columns,
            colors=consumption_colors,
            alpha=consumption_alpha,
            linewidth=0.15,
            edgecolor="white"
        )

        ax_consumption.set_ylabel("Energy demand [kWh]")
        ax_consumption.set_xlabel("Hour of the day")

        ax_consumption.grid(axis="y", linestyle="-", linewidth=0.4, alpha=0.35)
        ax_consumption.grid(axis="x", visible=False)
        ax_consumption.spines["top"].set_visible(False)
        ax_consumption.spines["right"].set_visible(False)

        consumption_handles, consumption_labels = ax_consumption.get_legend_handles_labels()
        ax_consumption.legend(
            consumption_handles,
            consumption_labels,
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            frameon=False,
            fontsize=consumption_legend_font_size,
            handlelength=1.2,
            borderaxespad=0.0
        )

        _format_hour_axis(ax_consumption, x, tick_interval_hours=2)

        if show_title:
            fig.suptitle(title, y=0.995, fontsize=title_font_size)

        os.makedirs(output_folder, exist_ok=True)

        output_path = os.path.join(output_folder, file_name)

        fig.savefig(
            output_path,
            dpi=600,
            bbox_inches="tight",
            pad_inches=0.04
        )

        if save_pdf:
            pdf_output_path = os.path.splitext(output_path)[0] + ".pdf"
            fig.savefig(
                pdf_output_path,
                bbox_inches="tight",
                pad_inches=0.04
            )

        plt.close(fig)

    return output_path

### 3.6.1 Functions for additional-load definition

In [62]:
if RUN_ADDITIONAL_LOADS:
    #-------------------------------------------------------------------
    #       Sec. 3.6 -- 001) FUNCTIONS FOR DEFINING HOUSEHOLD STRUCTURE
    #-------------------------------------------------------------------
    # The following functions identify which household types are present,
    # which members belong to each household, and how individual-level
    # quantities should be aggregated at household level.

    # Function to identify the household associated with each activity-profile column.
    def identify_households(activity_df):
        """
        Identify households and household types from the column names of the
        activity-profile DataFrame.

        Expected column format:
            "Household_(number)_Member_(number)_(season)_(household_type)"

        Returns
        -------
        tuple[pd.Series, pd.Series]
            - household identifiers, e.g. "Household_1";
            - canonical English household types.
        """
        extracted = activity_df.columns.to_series().str.extract(
            r'(Household_\d+)_Member_\d+_[^_]+_(.*)'
        )
        extracted.columns = ['Household', 'Household type']

        if extracted.isnull().any().any():
            raise ValueError(
                "Invalid column format in at least one activity-profile column."
            )

        return extracted['Household'], extracted['Household type'].map(canonical_household_type)


    # Function to distribute hourly values over 10-minute timesteps.
    def hourly_to_10min(hourly: pd.Series) -> pd.Series:
        # Forward-fill hourly values to 10-minute resolution and divide each hourly
        # value by 6 so that energy is conserved.
        return hourly.resample("10min").ffill() / 6


    # Function to infer whether a household member is at home.
    def infer_occupancy_hybrid_probabilistic(series: pd.Series, lookahead: int = 6) -> pd.Series:
        """
        Return a 0/1 occupancy flag for each timestep.

        Logic:
        - travel blocks, activity code 7, are always treated as away from home;
        - before the first travel block and after the last one, occupancy is inferred
          directly from the activity category at each timestep;
        - between two travel blocks, a single occupancy state is sampled for the whole
          interval using the average p_home value of the activities in that interval;
        - for selected activities, such as Eating and Phone use, p_home can depend on
          nearby Work activities.
        """
        vals = series.values
        n = len(vals)
        occ = np.full(n, np.nan, dtype=float)

        # Identify travel blocks.
        is_travel = vals == 7
        diffs = np.diff(is_travel.astype(int), prepend=0, append=0)
        starts = np.nonzero(diffs == 1)[0]
        ends = np.nonzero(diffs == -1)[0] - 1

        # 1) Before the first travel block: timestep-by-timestep logic.
        first_start = starts[0] if starts.size > 0 else n
        for i in range(0, first_start):
            code_i = vals[i]
            cat = activity_map.get(code_i, {'cat': 'maybe'})['cat']

            if cat == 'home':
                occ[i] = 1
            elif cat == 'away':
                occ[i] = 0
            else:
                occ[i] = occ[i - 1] if i > 0 else 1

        # 2) Travel blocks: always away from home.
        for block_start, block_end in zip(starts, ends):
            occ[block_start:block_end + 1] = 0

        # 3) Intervals between travel blocks: assign one occupancy state per interval.
        for idx in range(len(starts) - 1):
            seg_start = ends[idx] + 1
            seg_end = starts[idx + 1] - 1

            if seg_start > seg_end:
                continue

            p_home_values = []

            for j in range(seg_start, seg_end + 1):
                code_j = vals[j]

                # For Eating (3) and Phone use (22), check whether Work (14)
                # occurs in the following lookahead timesteps without an intervening
                # travel event.
                if code_j in (3, 22):
                    future = vals[j + 1:min(n, j + 1 + lookahead)]
                    p_home_j = 1.0

                    if 14 in future:
                        rel = list(future).index(14)
                        abs_work_idx = j + 1 + rel

                        if 7 not in vals[j + 1:abs_work_idx]:
                            p_home_j = activity_map[14]['p_home']
                else:
                    p_home_j = activity_map.get(code_j, {'p_home': 0.5})['p_home']

                p_home_values.append(p_home_j)

            interval_probability = sum(p_home_values) / len(p_home_values)

            # One Bernoulli draw for the whole interval.
            interval_state = np.random.binomial(1, interval_probability)
            occ[seg_start:seg_end + 1] = interval_state

        # 4) After the last travel block: timestep-by-timestep logic.
        if ends.size > 0 and ends[-1] < n - 1:
            for i in range(ends[-1] + 1, n):
                code_i = vals[i]
                cat = activity_map.get(code_i, {'cat': 'maybe'})['cat']

                if cat == 'home':
                    occ[i] = 1
                elif cat == 'away':
                    occ[i] = 0
                else:
                    occ[i] = occ[i - 1]

        # If no travel blocks are present, the first block already filled the series.
        # Fill any remaining NaN values for safety.
        occ = np.where(np.isnan(occ), 1, occ).astype(int)

        return pd.Series(occ, index=series.index)


    # Function to extract the household identifier from a person-level column name.
    def person_to_household(col_name):
        match = re.match(r'^(Household_\d+)_', col_name)
        return match.group(1) if match else None


    # Function to aggregate person-level values to household-level values.
    def aggregate_person_to_household_sum(person_series):
        household_values = {}

        for person, value in person_series.items():
            household_id = person_to_household(person)

            if household_id is None:
                continue

            household_values.setdefault(household_id, 0.0)
            household_values[household_id] += float(value)

        return pd.Series(household_values).sort_index()


    # Function to aggregate a person-indexed Series to the household index used downstream.
    def safe_get_person_series(person_series):
        household_series = aggregate_person_to_household_sum(person_series)
        household_series.index = household_series.index.to_series().str.extract(r'^(Household_\d+)')[0]
        household_series = household_series.reindex(household_index).fillna(0.0)

        return household_series


    #-------------------------------------------------------------------
    #       Sec. 3.6 -- 002) FUNCTIONS FOR DEFINING ADDITIONAL LOADS
    #-------------------------------------------------------------------

    # --------
    #  COOKING

    # Function to calculate induction-hob electricity consumption.
    def calculate_cooking_consumption_by_season(
        activity_df: pd.DataFrame,
        induction_ownership: dict,
        induction_use_probability: float,
        induction_consumption_kwh: float
    ) -> pd.DataFrame:
        """
        Return a person-level DataFrame of induction-hob electricity consumption.

        Parameters
        ----------
        activity_df : pd.DataFrame
            Activity-profile DataFrame with one column per household member.
        induction_ownership : dict
            Household-level dictionary indicating whether each household owns an
            induction hob.
        induction_use_probability : float
            Probability of using the induction hob when activity code 6, Cooking,
            occurs.
        induction_consumption_kwh : float
            Electricity consumption assigned to each cooking event.
        """
        df_cooking = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)

        for person in activity_df.columns:
            simple_household_id = re.match(r'(Household_\d+)', person).group(1)

            if not induction_ownership.get(simple_household_id, False):
                continue

            for ts, activity_code in activity_df[person].items():
                if activity_code == 6 and random.random() < induction_use_probability:
                    df_cooking.at[ts, person] = induction_consumption_kwh

        return df_cooking


    # Function to avoid or down-weight overlapping cooking events within households.
    def resolve_cooking_overlap_by_household(
        df_cook_person: pd.DataFrame,
        window_ts: int = 2,
        secondary_event_weight: float = 0.0,
        main_event_criterion: str = "max_energy"   # "max_energy" or "first"
    ) -> pd.DataFrame:
        """
        Post-process cooking consumption to avoid double counting among members
        of the same household.

        Logic:
        1) consecutive positive timesteps for each person are grouped into one
           cooking event;
        2) events close in time within the same household are grouped into clusters;
        3) within each cluster, one main event is kept at 100%, while the remaining
           events are either removed or down-weighted.
        """
        if df_cook_person.empty:
            return df_cook_person.copy()

        df_out = pd.DataFrame(
            0.0,
            index=df_cook_person.index,
            columns=df_cook_person.columns,
            dtype=float
        )

        # Map each person-level column to the corresponding household.
        household_of_person = df_cook_person.columns.to_series().str.extract(r'^(Household_\d+)')[0]

        for household_id in household_of_person.dropna().unique():
            cols = household_of_person[household_of_person == household_id].index.tolist()
            events = []

            # Build person-level cooking events as consecutive positive-consumption blocks.
            for person in cols:
                vals = df_cook_person[person].to_numpy(dtype=float)
                active = vals > 0

                if not active.any():
                    continue

                starts = np.flatnonzero(active & np.r_[True, ~active[:-1]])
                ends = np.flatnonzero(active & np.r_[~active[1:], True])

                for start, end in zip(starts, ends):
                    events.append({
                        "person": person,
                        "start": int(start),
                        "end": int(end),
                        "energy": float(vals[start:end + 1].sum())
                    })

            if not events:
                continue

            # Sort events and create time-close clusters.
            events.sort(key=lambda x: x["start"])

            clusters = []
            current_cluster = [events[0]]
            current_max_end = events[0]["end"]

            for event in events[1:]:
                # Same cluster if the event starts within window_ts from the
                # end of the active cluster.
                if event["start"] <= current_max_end + window_ts:
                    current_cluster.append(event)
                    current_max_end = max(current_max_end, event["end"])
                else:
                    clusters.append(current_cluster)
                    current_cluster = [event]
                    current_max_end = event["end"]

            clusters.append(current_cluster)

            # Within each cluster, keep one main event.
            for cluster in clusters:
                if main_event_criterion == "first":
                    main_event = min(cluster, key=lambda x: (x["start"], x["end"]))
                else:
                    main_event = max(cluster, key=lambda x: (x["energy"], -x["start"]))

                for event in cluster:
                    person = event["person"]
                    start, end = event["start"], event["end"]

                    src_col = df_cook_person.columns.get_loc(person)
                    dst_col = df_out.columns.get_loc(person)

                    segment = df_cook_person.iloc[start:end + 1, src_col].to_numpy(dtype=float)

                    weight = 1.0 if event is main_event else float(secondary_event_weight)

                    if weight > 0:
                        df_out.iloc[start:end + 1, dst_col] += weight * segment

        return df_out


    # Function to diagnose cooking overlaps within households.
    def diagnose_cooking_overlap_by_household(
        df_cook_person: pd.DataFrame,
        window_ts: int = 2
    ):
        """
        Diagnose cooking overlap within each household.

        Returns
        -------
        tuple
            - df_ts_events: timestep-level overlap events, where at least two members
              cook in the same timestep;
            - timestep_count_by_household: number of overlapping timesteps by household;
            - df_cluster_events: time-close cooking clusters involving at least two
              members of the same household;
            - cluster_count_by_household: number of overlapping clusters by household;
            - df_summary: compact household-level summary.
        """

        if df_cook_person.empty:
            empty_df = pd.DataFrame()
            empty_s = pd.Series(dtype=float)
            return empty_df, empty_s, empty_df, empty_s, empty_df

        # A) Timestep-level simultaneous-overlap diagnostics.
        cook_bin = (df_cook_person > 0).astype(int)
        household_of_person = cook_bin.columns.to_series().str.extract(r'^(Household_\d+)')[0]

        cooking_count_by_household = cook_bin.T.groupby(household_of_person).sum().T
        overlap_mask_ts = cooking_count_by_household >= 2

        ts_events = overlap_mask_ts.stack()
        ts_events = ts_events[ts_events]

        if len(ts_events) > 0:
            df_ts_events = ts_events.index.to_frame(index=False)
            df_ts_events.columns = ["Datetime", "Household"]
            df_ts_events["n_members_cooking"] = (
                cooking_count_by_household.stack().reindex(ts_events.index).values
            )
        else:
            df_ts_events = pd.DataFrame(columns=["Datetime", "Household", "n_members_cooking"])

        timestep_count_by_household = overlap_mask_ts.sum(axis=0).rename("n_timesteps_overlap")

        # B) Event-level and cluster-level diagnostics.
        all_cluster_rows = []
        cluster_count_dict = {}

        for household_id in household_of_person.dropna().unique():
            cols = household_of_person[household_of_person == household_id].index.tolist()
            events = []

            # Reconstruct person-level cooking events as consecutive positive blocks.
            for person in cols:
                vals = df_cook_person[person].to_numpy(dtype=float)
                active = vals > 0

                if not active.any():
                    continue

                starts = np.flatnonzero(active & np.r_[True, ~active[:-1]])
                ends = np.flatnonzero(active & np.r_[~active[1:], True])

                for start, end in zip(starts, ends):
                    events.append({
                        "Household": household_id,
                        "person": person,
                        "start_idx": int(start),
                        "end_idx": int(end),
                        "start_time": df_cook_person.index[start],
                        "end_time": df_cook_person.index[end],
                        "duration_ts": int(end - start + 1),
                        "energy_kWh": float(vals[start:end + 1].sum())
                    })

            if not events:
                cluster_count_dict[household_id] = 0
                continue

            events = sorted(events, key=lambda x: x["start_idx"])

            clusters = []
            current_cluster = [events[0]]
            current_end = events[0]["end_idx"]

            for event in events[1:]:
                if event["start_idx"] <= current_end + window_ts:
                    current_cluster.append(event)
                    current_end = max(current_end, event["end_idx"])
                else:
                    clusters.append(current_cluster)
                    current_cluster = [event]
                    current_end = event["end_idx"]

            clusters.append(current_cluster)

            # Keep only clusters involving at least two different people.
            n_overlap_clusters = 0

            for i, cluster in enumerate(clusters, start=1):
                people_involved = sorted(set(event["person"] for event in cluster))

                if len(people_involved) < 2:
                    continue

                n_overlap_clusters += 1

                start_idx_cluster = min(event["start_idx"] for event in cluster)
                end_idx_cluster = max(event["end_idx"] for event in cluster)

                all_cluster_rows.append({
                    "Household": household_id,
                    "cluster_id": f"{household_id}_cluster_{i}",
                    "start_cluster": df_cook_person.index[start_idx_cluster],
                    "end_cluster": df_cook_person.index[end_idx_cluster],
                    "cluster_duration_ts": int(end_idx_cluster - start_idx_cluster + 1),
                    "n_events_cluster": int(len(cluster)),
                    "n_people_involved": int(len(people_involved)),
                    "people_involved": " | ".join(people_involved),
                    "cluster_energy_kWh": float(sum(event["energy_kWh"] for event in cluster))
                })

            cluster_count_dict[household_id] = n_overlap_clusters

        df_cluster_events = pd.DataFrame(all_cluster_rows)
        cluster_count_by_household = pd.Series(cluster_count_dict).rename("n_cluster_overlap")

        # C) Household-level summary.
        idx = sorted(set(timestep_count_by_household.index).union(set(cluster_count_by_household.index)))
        df_summary = pd.DataFrame(index=idx)
        df_summary["n_timesteps_overlap"] = timestep_count_by_household.reindex(idx).fillna(0).astype(int)
        df_summary["n_cluster_overlap"] = cluster_count_by_household.reindex(idx).fillna(0).astype(int)

        if not cooking_count_by_household.empty:
            df_summary["max_simultaneous_members_per_ts"] = (
                cooking_count_by_household.max(axis=0).reindex(idx).fillna(0).astype(int)
            )
        else:
            df_summary["max_simultaneous_members_per_ts"] = 0

        return df_ts_events, timestep_count_by_household, df_cluster_events, cluster_count_by_household, df_summary


    # --------
    #  DHW

    # Function to calculate domestic hot water demand associated with showers.
    def calculate_dhw_consumption_by_season(
        activity_df: pd.DataFrame,
        shower_ownership: dict,
        morning_shower_use_probability: float,
        afternoon_shower_use_probability: float,
        min_consumption: float,
        max_consumption: float,
        eta_boiler: float
    ) -> pd.DataFrame:
        """
        For each timestep with activity code 2, Personal care & dressing:
        - use morning_shower_use_probability if ts.hour < 12;
        - use afternoon_shower_use_probability if ts.hour >= 12;
        - if the same person has already taken a morning shower on the same day,
          the afternoon shower probability is set to zero.
        """
        dhw_df = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)

        for person in activity_df.columns:
            simple_household_id = re.match(r'(Household_\d+)', person).group(1)

            if not shower_ownership.get(simple_household_id, False):
                continue

            # Track morning showers by date.
            morning_shower_dates = set()

            for ts, activity_code in activity_df[person].items():
                if activity_code != 2:
                    continue

                day = ts.date()

                if ts.hour < 12:
                    p_use = morning_shower_use_probability
                else:
                    # Skip afternoon showers if a morning shower already occurred.
                    if day in morning_shower_dates:
                        continue
                    p_use = afternoon_shower_use_probability

                if random.random() < p_use:
                    if ts.hour < 12:
                        morning_shower_dates.add(day)

                    person_consumption = random.uniform(
                        min_consumption / eta_boiler,
                        max_consumption / eta_boiler
                    )
                    dhw_df.at[ts, person] = person_consumption

        return dhw_df


    # Function to build DHW boiler specifications for each household.
    def build_dhw_boiler_specs_by_household(
        household_type_by_household: Dict[str, str],
        dhw_specs: Dict[str, Tuple]
    ) -> Dict[str, Dict[str, float]]:
        """
        Build the mapping Household_X -> DHW boiler specifications from the
        household type assigned to each household.

        Compatible with both 3-value dhw_specs entries:
            (equivalent_people, capacity_kwh_th, recharge_power_kw)

        and 4-value dhw_specs entries:
            (equivalent_people, capacity_kwh_th, recharge_power_kw, loss_rate_h)

        In Module 03, loss_rate_h is not used directly to build DHW demand.
        It is kept for downstream use in Module 04 / HAGRID.
        """
        boiler_specs = {}

        for household_id, household_type in household_type_by_household.items():
            if household_type not in dhw_specs:
                raise KeyError(f"Household type '{household_type}' not found in dhw_specs.")

            values = dhw_specs[household_type]

            if len(values) == 3:
                equivalent_people, capacity_kwh_th, recharge_power_kw = values
                loss_rate_h = None

            elif len(values) >= 4:
                equivalent_people, capacity_kwh_th, recharge_power_kw, loss_rate_h = values[:4]

            else:
                raise ValueError(
                    f"dhw_specs['{household_type}'] must contain at least 3 values: "
                    "(equivalent_people, capacity_kwh_th, recharge_power_kw). "
                    f"Found: {values}"
                )

            boiler_specs[household_id] = {
                "household_type": household_type,
                "equivalent_people": int(equivalent_people),
                "capacity_kwh_th": float(capacity_kwh_th),
                "recharge_power_kw": float(recharge_power_kw),
            }

            if loss_rate_h is not None:
                boiler_specs[household_id]["loss_rate_h"] = float(loss_rate_h)

        return boiler_specs


    # Function to shift simultaneous or time-close showers when boiler capacity is not plausible.
    def resolve_dhw_overlap_by_household(
        person_dhw_df: pd.DataFrame,
        dhw_boiler_specs_by_household: Dict[str, Dict[str, float]],
        simultaneous_probability: float,
        step_minutes: int = 10,
        alpha_util_boiler: float = 0.75,
        reference_shower_energy_kwh: float = None,
        use_average_shower_for_shift: bool = False,
        gap_reset_cluster_ts: int = 12,
        max_shift_steps: int = 72
    ) -> pd.DataFrame:
        """
        Post-process DHW demand to make shower timing more plausible without
        explicitly simulating the physical state of charge of the boiler.

        Logic:
        - for each household, the assigned boiler is used only as a plausibility
          constraint;
        - the nominal usable capacity is alpha_util_boiler * boiler_capacity;
        - the nominal recharge per timestep is boiler_power * dt;
        - all shower events in the household are processed chronologically;
        - if two or more showers are too close in time and their cumulative energy
          exceeds the plausible boiler budget, the next shower is shifted by the
          minimum number of timesteps compatible with recharge;
        - shifted showers are reinserted into the timeline and can be shifted again
          if they collide with other showers.

        Important:
        - this is not a physical boiler model;
        - it does not use initial SOC, thermal losses, hysteresis, prices, or
          operational constraints;
        - it only creates a plausible DHW demand profile for the downstream model.
        """
        if person_dhw_df.empty:
            return person_dhw_df.copy()

        df_out = pd.DataFrame(
            0.0,
            index=person_dhw_df.index,
            columns=person_dhw_df.columns,
            dtype=float
        )

        step_h = step_minutes / 60.0
        n_steps = len(person_dhw_df.index)

        household_of_person = person_dhw_df.columns.to_series().str.extract(r'^(Household_\d+)')[0]
        households = household_of_person.dropna().unique()

        for household_id in households:
            cols = household_of_person[household_of_person == household_id].index.tolist()

            if len(cols) == 0:
                continue

            if household_id not in dhw_boiler_specs_by_household:
                raise KeyError(f"Missing DHW boiler specifications for household '{household_id}'.")

            specs = dhw_boiler_specs_by_household[household_id]
            boiler_capacity_kwh_th = float(specs["capacity_kwh_th"])
            recharge_power_kw = float(specs["recharge_power_kw"])

            usable_cluster_budget_kwh = alpha_util_boiler * boiler_capacity_kwh_th
            charging_energy_per_step = recharge_power_kw * step_h

            # bucket[t] = list of shower events to process at timestep t.
            # Each item is (person, shower_energy, original_timestep).
            bucket = {}

            for person in cols:
                vals = person_dhw_df[person].to_numpy(dtype=float)
                nonzero_indices = np.flatnonzero(vals > 0)

                for t_idx in nonzero_indices:
                    bucket.setdefault(int(t_idx), []).append(
                        (person, float(vals[t_idx]), int(t_idx))
                    )

            # Active-cluster variables.
            cluster_energy_debt_kwh = 0.0
            last_accepted_ts = None

            for t_idx in range(n_steps):
                if t_idx not in bucket:
                    continue

                queue = bucket.pop(t_idx)

                # If enough time has passed since the last accepted shower, close the
                # previous cluster and reset the plausible energy debt.
                if last_accepted_ts is None:
                    cluster_energy_debt_kwh = 0.0
                else:
                    gap_ts = t_idx - last_accepted_ts

                    if gap_ts >= gap_reset_cluster_ts:
                        cluster_energy_debt_kwh = 0.0
                    else:
                        cluster_energy_debt_kwh = max(
                            0.0,
                            cluster_energy_debt_kwh - gap_ts * charging_energy_per_step
                        )

                accepted_showers_in_ts = 0

                while queue:
                    person, shower_energy, original_timestep = queue.pop(0)

                    # From the second shower in the same timestep onward, simultaneous
                    # showers are accepted only with simultaneous_probability.
                    simultaneity_not_allowed = (
                        accepted_showers_in_ts >= 1
                        and random.random() >= simultaneous_probability
                    )

                    if use_average_shower_for_shift and reference_shower_energy_kwh is not None:
                        shift_energy = reference_shower_energy_kwh
                    else:
                        shift_energy = shower_energy

                    exceeds_cluster_budget = (
                        cluster_energy_debt_kwh + shower_energy
                    ) > (usable_cluster_budget_kwh + 1e-12)

                    # Case 1: the shower can be accepted at the current timestep.
                    if (not simultaneity_not_allowed) and (not exceeds_cluster_budget):
                        df_out.iat[t_idx, df_out.columns.get_loc(person)] += shower_energy
                        cluster_energy_debt_kwh += shower_energy
                        last_accepted_ts = t_idx
                        accepted_showers_in_ts += 1
                        continue

                    # Case 2: the shower must be shifted.
                    shift_steps = 1 if simultaneity_not_allowed else 0

                    if exceeds_cluster_budget:
                        excess_kwh = max(
                            0.0,
                            cluster_energy_debt_kwh + shower_energy - usable_cluster_budget_kwh
                        )

                        if charging_energy_per_step > 0:
                            shift_for_budget = int(np.ceil(excess_kwh / charging_energy_per_step))
                        else:
                            shift_for_budget = 1

                        if charging_energy_per_step > 0:
                            shift_for_shower_recharge = int(np.ceil(shift_energy / charging_energy_per_step))
                        else:
                            shift_for_shower_recharge = 1

                        shift_steps = max(shift_steps, shift_for_budget, shift_for_shower_recharge)

                    shift_steps = max(1, shift_steps)

                    if max_shift_steps is not None:
                        shift_steps = min(shift_steps, max_shift_steps)

                    new_idx = t_idx + shift_steps

                    # If the shifted event exceeds the time horizon, place it at the
                    # last available timestep.
                    if new_idx >= n_steps:
                        new_idx = n_steps - 1

                    bucket.setdefault(new_idx, []).append(
                        (person, shower_energy, original_timestep)
                    )

        return df_out


    # --------
    #  EV

    # Function to calculate EV home-charging electricity consumption.
    def calculate_ev_consumption_by_season(
        activity_df: pd.DataFrame,
        df_occ: pd.DataFrame,
        ev_presence_by_household: dict,
        car_use_probability: float,
        km_per_step: float,
        # Consumption._per_km: float,
        consumption_by_household: dict,
        # battery_capacity_kwh: float,
        battery_capacity_by_household: dict,
        workplace_charging_probability: float,
        home_charging_power_kw: float,
        workplace_charging_power_kw: float,
        # initial_soc: float,
        charging_efficiency: float = 0.99,
        min_dwell_steps: int = 6,

        lookahead: int = 6
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """
        Returns:
        - df_charge_home_greedy: kWh of home charging 
            (this already represents the EV battery charging profile 
            accounting for occupancy -> maximum charging
            as soon as charging is possible)
        - df_charge_work: kWh of workplace charging
        - df_soc: battery state of charge [%] at each timestep
        """
        # Hours per timestep.
        step_h = 10/60
        # Energy charged at home and at work.
        home_charging_per_step   = home_charging_power_kw   * step_h * charging_efficiency
        workplace_charging_per_step = workplace_charging_power_kw * step_h * charging_efficiency
        
        # Initialize DataFrames.
        df_charge_home_greedy = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)
        df_charge_work = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)
        df_soc         = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)
        df_km         = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)
        ev_demand_df = pd.DataFrame(0.0, index=activity_df.index, columns=activity_df.columns)




        n_ts = len(activity_df)
        # ===== Person-level pre-assignments, performed once =====
        initial_soc_by_person = {}
        energy_to_full_by_person = {}
    
        # Build metadata rows using the activity_df column names, e.g. 'Household_1_Member_1_...').
        ev_metadata_rows = []

        for person in activity_df.columns:
            # Consider only Member_1, consistently with the rest of the code. Remove this filter to include all members.
            if not re.match(r'Household_\d+_Member_1_', person):
                continue

            simple_household_id = re.match(r'(Household_\d+)', person).group(1)
            has_ev = ev_presence_by_household.get(simple_household_id, False)
            cap = battery_capacity_by_household.get(simple_household_id, None)
            km_consumption = consumption_by_household.get(simple_household_id, None)
            if has_ev and (cap is not None):
                # soc0 = float(random.uniform(0.95,1))
                soc0 = 1 # Assume all cars are initially fully charged, otherwise a simultaneous peak appears at the beginning of the week.
                initial_soc_by_person[person] = soc0
                energy_to_full_by_person[person] = (1.0 - soc0) * cap

            else:
                initial_soc_by_person[person] = np.nan
                energy_to_full_by_person[person] = 0.0

            ev_metadata_rows.append({
                "person": person,
                "household": simple_household_id,
                "has_ev": bool(has_ev),
                "bev_capacity_kWh": cap,
                "cons_kWh_per_km": km_consumption,
                "soc_initial": initial_soc_by_person.get(person, np.nan),
                "energy_to_full_kWh": energy_to_full_by_person.get(person, 0.0)

            })

        for person in activity_df.columns:
            # Only Member_1.
            if not re.match(r'Household_\d+_Member_1_', person):
                continue

            # Check whether the household owns an EV.
            simple_household_id = re.match(r'(Household_\d+)', person).group(1)
            if not ev_presence_by_household.get(simple_household_id, False):
                continue

            # Retrieve household-specific EV parameters.
            battery_capacity_kwh = battery_capacity_by_household[simple_household_id]
            consumption_per_km   = consumption_by_household[simple_household_id]    

            # Initial state of charge. 
            # initial_soc = random.uniform(0.95,1) # random, otherwise a large demand peak appears for all EVs at the start of the week
            # soc = initial_soc * battery_capacity_kwh  # Initial SoC in kWh
            soc = initial_soc_by_person[person] * battery_capacity_kwh
            dwell = 0                        # contatore dei passi a home consecutivi
            work_block = False             # whether the EV is currently in a workplace-charging block


            vals     = activity_df[person].values
            occ_vals = df_occ[person].values

            for t in range(n_ts):
                # First, record the SoC as a percentage.
                df_soc.iat[t, df_soc.columns.get_loc(person)] = soc/battery_capacity_kwh*100

                code = vals[t]
                
                # 1) If travelling -> driving consumption.
                if code == 7:
                    dwell = 0
                    work_block = False
                    if random.random() < car_use_probability:
                        # Calculate electricity consumed while travelling.
                        e_drive = km_per_step * consumption_per_km
                        ev_demand_df.iat[t, ev_demand_df.columns.get_loc(person)] = e_drive                      
                        
                        # Consumption.
                        soc = max(0.0, soc - km_per_step*consumption_per_km)
                        # Distance travelled in each timestep.
                        df_km.iat[t, df_km.columns.get_loc(person)] = km_per_step        
                    
                    continue

                # 2) If the activity is Work and there is no travel in the next lookahead timesteps.
                if code == 14:
                    dwell = 0
                    # Check that there is no immediate travel.
                    next_slice = vals[t+1:t+1+lookahead]
                    if 7 not in next_slice and random.random()<workplace_charging_probability:
                        work_block = True
                    else:
                        work_block = False

                # 3) If the EV is in a workplace-charging block.
                if work_block:
                    if soc < battery_capacity_kwh:
                        df_charge_work.iat[t, df_charge_work.columns.get_loc(person)] = workplace_charging_per_step
                        soc = min(battery_capacity_kwh, soc + workplace_charging_per_step)
                    # Exit the workplace-charging block as soon as the activity changes.
                    if code != 14:
                        work_block = False
                    continue

                # 4) Otherwise, count consecutive timesteps at home.
                if occ_vals[t]:
                    dwell += 1
                else:
                    dwell = 0


                # If the minimum dwell time at home is reached and the battery is not full, charge at home.
                if dwell >= min_dwell_steps and soc < battery_capacity_kwh:
                    df_charge_home_greedy.iat[t, df_charge_home_greedy.columns.get_loc(person)] = home_charging_per_step
                    soc = min(battery_capacity_kwh, soc + home_charging_per_step)

            # end loop t
            # # Complete df_soc with the last value for possible NaN values.
            # df_soc[person].ffill(inplace=True)
        # More efficient alternative without FutureWarning.
        # Fill all df_soc NaN values with forward-fill in one vectorized operation.
        df_soc = df_soc.ffill()


        # ===== EV diagnostics and metadata export =====
        ev_metadata_df = pd.DataFrame(ev_metadata_rows).set_index("person")
        # Save. file (usa "additional_consumption_output_xls_folder" definito nello scope globale)
        try:
            out_meta = os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_ev_metadata_by_person.xlsx')
            ev_metadata_df.to_excel(out_meta)
        except Exception:
            # Do not stop the simulation for a metadata-writing error.
            print("[WARN] Unable to save EV metadata. Check additional_consumption_output_xls_folder.")

        return df_charge_home_greedy, df_charge_work, df_soc, df_km, ev_demand_df



    # ################## OPTIONAL HELPER: uncomment if the normalized thermal-demand profile must be rebuilt, e.g. when changing the reference year
    # # Function to build the normalized thermal profile for the reference dwellings
    # def build_normalized_thermal_profile(
    #     heating_coefficients_path: str,
    #     temperature_data_path: str,
    #     thermal_profile_cop_excel_path: str
    # ) -> pd.Series:
    #     """
    #     Return an hourly 8760-row DataFrame:
    #       - `fabb_unitario`: unit thermal demand [kWh_th], linear T->Y
    #       - `fabb_norm`: normalized demand, sum=1 over the heating period
    #       - `COP`: hourly coefficient of performance
    #     and save it to Excel.
    #     """

    #     # 1) Load coefficients, temperature, and COP.
    #     coeff_df = pd.read_excel(heating_coefficients_path, index_col=0)
    #     temp_df  = (
    #         pd.read_excel(temperature_data_path, index_col=0, parse_dates=True)
    #           .rename(columns={"Temperatura [°C]":"T", "COP":"COP"})
    #           [["T","COP"]]
    #     )
    #     idx = temp_df.index
    #     hours = idx.hour

    #     # 2) Extract vectors a and b, 24 values each.
    #     a = coeff_df.loc["Angular coeff"].values  # shape (24,)
    #     b = coeff_df.loc["Intercept"].values      # shape (24,)

    #     # 3) Identify the heating period, 15 Nov-14 Apr.
    #     m, d = idx.month, idx.day
    #     heating_period = (
    #         ((m == 11) & (d >= 15)) |
    #         (m == 12) |
    #         (m <= 3) |
    #         ((m == 4) & (d <= 15))
    #     )
    #     mask_temp = temp_df["T"] <= 16

    #     # 4) Calculate unit hourly demand.
    #     Y = a[hours] * temp_df["T"].values + b[hours]
    #     Y[~(heating_period & mask_temp)] = 0
    #     Y = np.clip(Y, 0, None)

    #     # 5) Normalize over the heating period.
    #     total = Y.sum()
    #     Y_norm = Y / total if total > 0 else Y

    #     # 6) Build the DataFrame.
    #     df = pd.DataFrame({
    #         "fabb_unitario_th": Y,
    #         "fabb_norm":        Y_norm,
    #         "COP":              temp_df["COP"].values
    #     }, index=idx)

    #     # 7) Save to Excel.
    #     df.to_excel(thermal_profile_cop_excel_path, sheet_name="thermal_profile")

    #     return df
    # ##############################################################################



    # --------
    #  HEATING

    # Function to calculate annual thermal demand for heating for each household, once.
    def calculate_annual_thermal_demand_by_household(
        household_type_by_household: Dict[str,str],
        household_apartment_specs: Dict[str, Tuple[int,float,float]],
        min_thermal_demand_per_m2: float,
        max_thermal_demand_per_m2: float,
        seed: int = None
    ) -> Dict[str, float]:
        """
        Calculate and return a dictionary { 'Household_X': annual_thermal_demand_kWh_th }
        using the same logic previously used inside calculate_heating_consumption_by_season.
        If a seed is provided, set np.random.seed(seed) to make random generation reproducible.
        """
        if seed is not None:
            np.random.seed(seed)

        annual_thermal_demand = {}
        for household_id, ftype in household_type_by_household.items():
            if ftype not in household_apartment_specs:
                raise KeyError(f"Household type '{ftype}' not found")
            _, a_min, a_max = household_apartment_specs[ftype]
            area = np.random.uniform(a_min, a_max)
            consumption_per_m2 = np.random.uniform(min_thermal_demand_per_m2, max_thermal_demand_per_m2)
            annual_thermal_demand[household_id] = area * consumption_per_m2  # kWh_th/year

        return annual_thermal_demand



    # Function to calculate heat-pump electricity consumption for heating.
    def calculate_heating_consumption_by_season(
        thermal_profile_cop_excel_path: str,
        annual_thermal_demand: Dict[str, float],
        season: str,
        season_periods: Dict[str, Tuple[str,str]],
        df_occ: pd.DataFrame,             # 10-minute occupancy for all members
        min_absent_steps: int        # numero minimo di passi consecutivi fuori home

    ) -> pd.DataFrame:
        """
        Return a 10-minute DataFrame with heating electricity demand, kWh_el, for each household and season.
        It now takes annual_thermal_demand as input, calculated once outside the function.
        - thermal_profile_cop_excel_path: Excel path with sheet "thermal_profile" or legacy "profilo_termico", with columns:
            * 'fabb_norm' (8760h, sum=1)
            * 'COP'       (hourly)
        - annual_thermal_demand: dict { 'Household_X': kWh_th/year } PRE-CALCOLATO
        - season: season name (chiave in season_periods e cop_pdc)
        - season_periods: season -> (start_datetime, end_datetime)
        - df_occ: member presence at 10-minute resolution
        """
        # 1) read profile hourly normalizzato + COP
        # Read the normalized hourly thermal profile and COP.
        # The English sheet name is preferred, while the legacy Italian sheet name is
        # retained as a fallback for compatibility with older input files.
        try:
            df_prof = pd.read_excel(
                thermal_profile_cop_excel_path,
                sheet_name="thermal_profile",
                index_col=0,
                parse_dates=True
            )
        except ValueError:
            df_prof = pd.read_excel(
                thermal_profile_cop_excel_path,
                sheet_name="profilo_termico",
                index_col=0,
                parse_dates=True
            )

        df_prof = df_prof.loc[:, ['fabb_norm', 'COP']]

        # 2) Restrict to the selected seasonal period.
        start, end = season_periods[season]
        df_prof = df_prof.loc[start:end]

        # 3) Distribute annual demand over hourly timesteps.
        # DataFrame with hourly resolution and N households.
        df_annual_th = pd.DataFrame(
            { household_id: df_prof['fabb_norm'] * annual_estimate
            for household_id, annual_estimate in annual_thermal_demand.items() },
            index=df_prof.index
        )

        # 4) Create a 10-minute index and split each hour into six 10-minute steps.
        idx_10min = df_occ.index
        # idx_10min = pd.date_range(start=df_annual_th.index[0],
        #                           end=df_annual_th.index[-1],
        #                           freq='10T')
        df_annual_10min = (
            df_annual_th
            .reindex(idx_10min, method='ffill')   # spreading
            / 6                                     # kWh_th ogni 10'
        )

        # 5) Convert hourly COP to 10-minute resolution with forward-fill.
        # cop_10min = df_prof['COP'].reindex(idx_10min, method='ffill').fillna(method='ffill') # Deprecated; kept as legacy reference.
        # Reindex + forward-fill; use back-fill for possible initial NaN values.
        cop_10min = df_prof['COP'].reindex(idx_10min).ffill().bfill()

        # 6) Convert to electricity: kWh_el = kWh_th / COP.
        df_heating_el_10min = df_annual_10min.div(cop_10min, axis=0)    
        
        # 7) For each household, derive at-home presence: 
        #    from df_occ, with columns for each member, create a binary household series equal to 1 if at least one member is at home.
        household_presence = {}
        for household_id in household_type_by_household:
            # Select all members of that household.
            cols = [c for c in df_occ.columns if c.startswith(household_id + "_")]
            # For each timestep, 1 if at least one member is at home.
            household_presence[household_id] = df_occ[cols].max(axis=1)
        household_presence_df = pd.DataFrame(household_presence, index=df_occ.index)

        # 8) Identify runs where all members are absent for each household.
        absence_df = 1 - household_presence_df  # 1 = all absent, 0 = at least one member at home
        df_heat_el_masked = df_heating_el_10min.copy()

        for household_id in absence_df.columns:
            s = absence_df[household_id]
            # Label constant True/False blocks.
            block_id = (s != s.shift(fill_value=0)).cumsum()
            # For each block, calculate its length and whether it is an absence block.
            lengths = s.groupby(block_id).transform('sum')
            # Here lengths[i] is the number of consecutive timesteps equal to s[i], counting only True values.
            # Select all points where s==1 and the block length is >= min_absent_steps.
            to_zero = (s == 1) & (lengths >= min_absent_steps)
            df_heat_el_masked.loc[to_zero, household_id] = 0.0  
            
        return df_heat_el_masked
    #---------------------------------------------------------------------------------------------------------------------------------------------




    #-------------------------------------------------------------------
    #       Sec. 3.6 -- 003) FUNCTIONS FOR AGGREGATING ADDITIONAL LOADS
    #-------------------------------------------------------------------

    def persons_df_to_household_df(
        df_person: pd.DataFrame,
        household_index: pd.Index
    ) -> pd.DataFrame:
        """
        Convert a person-level DataFrame into a household-level DataFrame.

        Parameters
        ----------
        df_person : pd.DataFrame
            DataFrame with one column per person, e.g.
            "Household_1_Member_1_...".
        household_index : pd.Index
            Ordered household index to use in the output columns.

        Returns
        -------
        pd.DataFrame
            DataFrame with the same index as df_person and one column per household.
            Missing households are filled with zeros.
        """
        if df_person.shape[1] == 0:
            return pd.DataFrame(
                0.0,
                index=df_person.index,
                columns=household_index,
                dtype=float
            )

        cols = df_person.columns.to_series()
        household_series = cols.str.extract(r'^(Household_\d+)')[0]

        valid_mask = household_series.notna()

        if not valid_mask.any():
            return pd.DataFrame(
                0.0,
                index=df_person.index,
                columns=household_index,
                dtype=float
            )

        df_valid = df_person.loc[:, valid_mask.index[valid_mask]]
        household_for_valid_columns = household_series.loc[df_valid.columns]

        # Build a binary person-to-household matrix and aggregate via matrix product.
        aggregation_matrix = (
            pd.get_dummies(household_for_valid_columns)
            .reindex(columns=sorted(household_for_valid_columns.unique()))
            .astype(float)
        )

        arr = df_valid.values.dot(aggregation_matrix.values)

        household_df = pd.DataFrame(
            arr,
            index=df_valid.index,
            columns=aggregation_matrix.columns,
            dtype=float
        )

        household_df = household_df.reindex(columns=household_index, fill_value=0.0)

        return household_df


    #---------------------------------------------------------------------------------------------------------------------------------------------

## 3.7 Additional-load calculation
This section performs the actual calculation of each additional-load component: cooking, heating, cooling, EV charging, and DHW.  
It also generates weekly aggregated profiles and selected plots.

In [63]:
if RUN_ADDITIONAL_LOADS:
    # -----------------------------------
    # 1) Identify households and assign technologies.
    # This is done once and reused across all seasons.
    # -----------------------------------

    # Load the first season and identify households. 
    first_season = selectable_periods[0]
    initial_activity_df = pd.read_excel(os.path.join(household_profiles_input_dir, f"{household_profiles_name}.xlsx"),
                            sheet_name=first_season)
    initial_activity_df['Datetime'] = pd.to_datetime(initial_activity_df['Datetime'])
    initial_activity_df.set_index('Datetime', inplace=True)
    initial_activity_df = normalize_household_profile_columns(initial_activity_df, default_season=first_season)

    # Derive simple household identifiers from initial_activity_df.
    households0, household_types0 = identify_households(initial_activity_df)


    # Build the household -> household type dictionary.
    household_type_by_household = { household_id: canonical_household_type(household_types0[col]) 
                         for col,household_id in households0.items() }

    # Get the base person identifiers, e.g. Household_1_Member_1.
    base_person_ids = (
        initial_activity_df.columns
               .to_series()
               .str.extract(r'^(Household_\d+_Member_\d+)_')[0]
               .unique()
               .tolist()
    )

    simple_households = sorted(set(households0.values))

    # Build DHW boiler specifications for each household.
    dhw_boiler_specs_by_household = build_dhw_boiler_specs_by_household(
        household_type_by_household,
        dhw_specs
    )

    # Assign shower availability.
    shower_presence_by_household = {
        household_id: (random.random() < shower_ownership_probability)
        for household_id in simple_households
    }
    
    # Assign induction-hob availability.
    induction_presence_by_household = {
        household_id: (random.random() < induction_ownership_probability)
        for household_id in simple_households
    }   
    
    # Assign EV availability.
    ev_presence_by_household = {
      household_id: (random.random() < ev_ownership_probability)
      for household_id in simple_households
    }
    
    # Assign one EV class to each household that owns an EV.
    ev_class_by_household = {}
    battery_capacity_by_household = {}
    consumption_by_household = {}
    for household_id in simple_households:
        if ev_presence_by_household[household_id]:
            possible_ev_classes = ev_classes_by_household_type.get(household_type_by_household[household_id], list(ev_classes.keys()))
            selected_ev_class = random.choice(possible_ev_classes)
            ev_class_by_household[household_id]         = selected_ev_class
            battery_capacity_by_household[household_id]    = ev_classes[selected_ev_class]["capacity"]
            consumption_by_household[household_id]     = ev_classes[selected_ev_class]["cons_kwh_per_km"]
        else:
            ev_class_by_household[household_id]      = None
            battery_capacity_by_household[household_id] = None
            consumption_by_household[household_id]  = None
    
    # Initialize the weekly-distance accumulator.
    weekly_km = pd.Series(0.0, index=base_person_ids)


    # Pre-calculate 10-minute cooling profiles.
    cooling_profiles_10min = {}
    for season, (start, end) in season_periods.items():
        # 1) Build the hourly index for one week, 168 hours.
        hourly_idx = pd.date_range(start=start, periods=168, freq="h") # Lowercase "h" avoids deprecated uppercase "H".
        # 2) Map the hourly series.
        s_hourly = pd.Series(cooling_week_df[find_compatible_column(cooling_week_df, season, COOLING_SEASON_COLUMN_ALIASES)].values,
                             index=hourly_idx,
                             name="cooling_per_person_kW")
    
        # 3) Generate the 10-minute index directly, 1008 timesteps.
        idx_10min = pd.date_range(start=start, periods=7 * 24 * 6, freq="10min") # "10T" is deprecated; use "10min".
        # 4) Forward-fill hourly values and divide by 6.
        s_10min = s_hourly.reindex(idx_10min, method="ffill") / 6
    
        cooling_profiles_10min[season] = s_10min


    # Pre-calculate the normalized heating profile.
    # #########    DA DECOMMENTARE SE SI VUOLE COSTRUIRE IL PROFILO NORMALIZZATO DI FABBISOGNO TERMICO (es se si cambia anno osservazione) 
    # thermal_profile_df = build_normalized_thermal_profile(heating_coefficients_path, temperature_data_path, thermal_profile_cop_excel_path)
    # ##############################
    
    # Calculate annual thermal demand for heating.
    seed_for_heating = np.random.randint(0, 2**31 - 1)
    annual_thermal_demand_th = calculate_annual_thermal_demand_by_household(
        household_type_by_household,
        household_apartment_specs,
        min_thermal_demand_per_m2,
        max_thermal_demand_per_m2,
        seed=seed_for_heating
    )
    # Build a stable household index.
    household_index = pd.Index(sorted(annual_thermal_demand_th.keys()))
    
    
    
    
    
    # Before the seasonal loop, prepare containers for seasonal DataFrames.
    dfs_by_season = {}   # key = season_name -> household_season_df
    annual_cooling_by_household = None  # set during summer
    soc_dfs_by_season = {}
    household_dhw_dfs_by_season = {}       # used to inspect simultaneous showers within the same household

    # -----------------------------------
    # 2) Prepare the ExcelWriter objects, one for each output: DHW, cooking, occupancy, etc.
    # -----------------------------------
    writer_occ   = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_occupancy.xlsx'), engine='openpyxl')
    dhw_writer   = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_DHW_consumption_kWh.xlsx'),        engine='openpyxl')
    induction_writer   = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_induction_consumption_kWh.xlsx'),      engine='openpyxl')
    
    # "ev_home_charging_greedy_writer" stores the greedy EV home-charging profile, accounting for occupancy and charging as soon as possible.
    ev_home_charging_greedy_writer   = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_EV_home_greedy_charging_kWh.xlsx'), engine='openpyxl')
    
    # This stores the actual electricity demand required by cars when travelling. It does not indicate when EVs are charged at home; home charging is optimized downstream.
    ev_demand_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_EV_electricity_demand_kWh.xlsx'), engine='openpyxl')

    # "writer_EV_charge_work" stores the possible greedy/dump EV workplace-charging profile.
    ev_work_charging_writer  = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_EV_work_greedy_charging_kWh.xlsx'), engine='openpyxl')
    ev_soc_writer  = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_SoC_EV.xlsx'), engine='openpyxl')
    ev_km_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_EV_km_travelled.xlsx'), engine='openpyxl')

    cooling_writer   = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_Cooling_consumption_kWh.xlsx'), engine='openpyxl')
    heating_writer   = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_Heating_consumption_kWh.xlsx'), engine='openpyxl')

    additional_total_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_weekly_aggregated_ts_kWh_additional_loads.xlsx'), engine='openpyxl')
    household_annual_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_annual_values_kWh_additional_loads.xlsx'), engine='openpyxl')

    household_timeseries_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_timeseries_kWh_per_household_TOT.xlsx'), engine='openpyxl')

    household_load_timeseries_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_ts_kWh_el_by_household_by_load.xlsx'), engine='openpyxl')
    dhw_overlap_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder, f'{num_households}_households_{selected_region}_DHW_overlap.xlsx'), engine='openpyxl')
    cooking_overlap_writer = pd.ExcelWriter(os.path.join(additional_consumption_output_xls_folder,f'{num_households}_households_{selected_region}_Cooking_overlap.xlsx'), engine='openpyxl')


    # -----------------------------------
    #   Sec. 3.7 -- 2b) Read all activity sheets before the seasonal loop, more efficient.
    # -----------------------------------
    # Read all profile sheets from the workbook into an in-memory dictionary.
    path_profiles = os.path.join(household_profiles_input_dir, f"{household_profiles_name}.xlsx")
    sheets_dict = pd.read_excel(path_profiles, sheet_name=None)  # {sheet_name: df}



    # ------------------------------------------------------------
    #   Sec. 3.7 -- 2c) Base-load file for the stacked conventional+additional plot
    # ------------------------------------------------------------
    base_load_xls = None

    if PLOT_STACKED_CONV_PLUS_ADDITIONAL:
        base_load_path = _resolve_base_load_excel_path()

        if base_load_path is not None:
            base_load_xls = pd.ExcelFile(base_load_path)
            print(f"[INFO][BASE LOAD] Loaded base-consumption file: {base_load_path}")



    # -----------------------------------
    #   Sec. 3.7 -- 3) LOOP OVER THE FOUR SEASONS
    # -----------------------------------
    for season in selectable_periods:
        
        # # Load the activity profile for the current season.
        # activity_df = pd.read_excel(os.path.join(household_profiles_input_dir, f"{household_profiles_name}.xlsx"),
        #                        sheet_name=season)
        # Load the activity profile for the current season from the pre-read sheets, which is more efficient.
        activity_sheet_name = next((s for s in sheets_dict if canonical_season(s) == season), season)
        activity_df = sheets_dict[activity_sheet_name].copy()

        activity_df['Datetime'] = pd.to_datetime(activity_df['Datetime'])
        activity_df.set_index('Datetime', inplace=True)
        activity_df = normalize_household_profile_columns(activity_df, default_season=season)

        # 3.1) -------- Domestic occupancy --------
        df_occ = activity_df.apply(infer_occupancy_hybrid_probabilistic)
        df_occ.to_excel(writer_occ, sheet_name=season)



        # 3.2) -------- DHW consumption, showers --------
        # First-pass DHW consumption, used to inspect how many simultaneous showers occur.
        raw_dhw_week_df = calculate_dhw_consumption_by_season(
            activity_df, 
            shower_presence_by_household, 
            morning_shower_use_probability,
            afternoon_shower_use_probability,
            min_shower_consumption_kwh_th,
            max_shower_consumption_kwh_th,
            dhw_boiler_efficiency
        )

        # Post-process DHW overlaps by shifting simultaneous or time-close shower events when the boiler energy budget is not plausible.
        # Make shower timing plausible based on boiler size and recharge power, without simulating the actual boiler SOC.
        dhw_week_df = resolve_dhw_overlap_by_household(
            raw_dhw_week_df,
            dhw_boiler_specs_by_household=dhw_boiler_specs_by_household,
            simultaneous_probability=simultaneous_shower_probability,
            step_minutes=step_minutes,
            alpha_util_boiler=alpha_util_boiler,
            reference_shower_energy_kwh=reference_shower_consumption_kwh_th,
            use_average_shower_for_shift=use_average_shower_for_shift,
            gap_reset_cluster_ts=gap_reset_cluster_ts,
            max_shift_steps=max_shower_shift_steps
        )
        # dhw_week_df = resolve_dhw_overlap_by_household(
        #     raw_dhw_week_df,
        #     simultaneous_probability=simultaneous_shower_probability,
        #     max_shift_steps=6
        # )

        dhw_week_df.to_excel(dhw_writer, sheet_name=season)

        # 3.2.1) --- Count simultaneous DHW events among members of the same household ---
        
        dhw_binary = (raw_dhw_week_df > 0).astype(int)                                             # 1) DHW consumption for a person -> value = 1
        household_of_person = dhw_binary.columns.to_series().str.extract(r'^(Household_\d+)')[0]            # 2) Map person -> household
        dhw_count_by_household = dhw_binary.groupby(household_of_person, axis=1).sum()                         # 3) Sum by household: number of members with DHW consumption at each timestep
        overlap_mask = dhw_count_by_household >= 2                                                    # 4) Overlap mask.
        events = overlap_mask.stack()                                                           # 5) Build the long-format event table: (Datetime, Household, members_with_DHW).
        events = events[events]  # Keep only True values.
        df_events = events.index.to_frame(index=False)
        df_events.columns = ["Datetime", "Household"]
        df_events["members_with_DHW"] = dhw_count_by_household.stack().reindex(events.index).values

        # 3.2.2) Define the simultaneous-DHW count table by household.
        count_by_household = overlap_mask.sum(axis=0).rename("n_timesteps_overlap")
        df_events.to_excel(dhw_overlap_writer, sheet_name=f"{season}_events_pre_process", index=False)
        count_by_household.to_frame().to_excel(dhw_overlap_writer, sheet_name=f"{season}_counts_pre_process")



        # 3.3) -------- Cooking consumption, induction hob --------
        raw_induction_week_df = calculate_cooking_consumption_by_season(
            activity_df,
            induction_presence_by_household,
            cold_season_induction_use_probability,
            induction_consumption_kwh,
        )

        # diagnostica PRE-process
        df_cook_ts_events, cook_ts_count, df_cook_cluster_events, cook_cluster_count, df_cook_summary = (
            diagnose_cooking_overlap_by_household(
                raw_induction_week_df,
                window_ts= cooking_window_steps
            )
        )

        df_cook_ts_events.to_excel(cooking_overlap_writer, sheet_name=f"{season}_events_ts_pre", index=False)
        cook_ts_count.to_frame().to_excel( cooking_overlap_writer, sheet_name=f"{season}_counts_ts_pre")
        df_cook_cluster_events.to_excel(cooking_overlap_writer, sheet_name=f"{season}_cluster_pre", index=False)
        df_cook_summary.to_excel(cooking_overlap_writer, sheet_name=f"{season}_summary_pre")

        # Post-process cooking.
        induction_week_df = resolve_cooking_overlap_by_household(
            raw_induction_week_df, 
            window_ts=cooking_window_steps, 
            secondary_event_weight=simultaneous_cooking_probability,
            main_event_criterion="max_energy")

        # Post-process diagnostics, after resolving cooking overlaps.
        df_cook_ts_events_post, cook_ts_count_post, df_cook_cluster_events_post, cook_cluster_count_post, df_cook_summary_post = (
            diagnose_cooking_overlap_by_household(induction_week_df, window_ts=cooking_window_steps))


        df_cook_ts_events_post.to_excel(cooking_overlap_writer, sheet_name=f"{season}_events_ts_post",index=False)
        cook_ts_count_post.to_frame().to_excel(cooking_overlap_writer, sheet_name=f"{season}_counts_ts_post")
        df_cook_cluster_events_post.to_excel(cooking_overlap_writer, sheet_name=f"{season}_cluster_post", index=False)
        df_cook_summary_post.to_excel(cooking_overlap_writer, sheet_name=f"{season}_summary_post")

        # Save the processed cooking profiles accounting for household-level simultaneity.
        induction_week_df.to_excel(induction_writer, sheet_name=season)


        
        # 3.4) -------- EV charging and driving demand --------
        df_charge_home_week, df_charge_work_week, df_soc, df_km, ev_demand_df = calculate_ev_consumption_by_season(
            activity_df, 
            df_occ, 
            ev_presence_by_household,
            car_use_probability, 
            km_per_step,
            # Consumption._per_km,
            consumption_by_household,
            # battery_capacity_kwh, 
            battery_capacity_by_household,
            workplace_charging_probability,
            home_charging_power_kw, 
            workplace_charging_power_kw,
            # initial_soc,
            charging_efficiency,
            min_dwell_steps,
            # lookahead
        )
        
        # 3.4.1) Keep only Member_1 columns belonging to households with an EV.
        cols_series = df_soc.columns.to_series()        # versione OTTIMIZZATA che evita FutureWARNINGS
        is_m1 = cols_series.str.contains(r'_Member_1_')
        household_for_column = cols_series.str.extract(r'^(Household_\d+)')[0]
        has_ev_mask = household_for_column.map(ev_presence_by_household).fillna(False).astype(bool)
        ev_soc_column = cols_series[is_m1 & has_ev_mask].tolist()

        
        # 3.4.2) For distance, reuse the same columns and optionally drop columns with zero km.
        km_sums = df_km[ev_soc_column].sum(axis=0)        # versione OTTIMIZZATA che evita FutureWARNINGS
        ev_km_column = km_sums[km_sums > 0].index.tolist()
        
        # 3.4.3) Filter the DataFrames.
        df_soc_filtrato = df_soc.loc[:, ev_soc_column]
        df_km_filtrato  = df_km.loc[:, ev_km_column]
        
        # 3.4.4) Write only the filtered DataFrames to Excel.
        df_soc_filtrato.to_excel(ev_soc_writer, sheet_name=season)
        df_km_filtrato.to_excel(ev_km_writer,  sheet_name=season)   
        df_charge_home_week.to_excel(ev_home_charging_greedy_writer, sheet_name=season)
        df_charge_work_week.to_excel(ev_work_charging_writer, sheet_name=season)

        ev_demand_week_df = ev_demand_df.copy()
        ev_demand_week_df.to_excel(ev_demand_writer, sheet_name=season)


        # Store the filtered SoC for the current season.
        soc_dfs_by_season[season] = df_soc_filtrato.copy()


        df_km_base = df_km_filtrato.rename(
            columns=lambda c: re.match(r'^(Household_\d+_Member_\d+)_', c).group(1)
            )

        # Sum weekly km for each base person identifier.
        weekly_km += df_km_base.sum()



        # 3.5) -------- Cooling consumption --------
        # 3.5.1) Get the start timestamp from the occupancy series.
        start_ts = df_occ.index.min().normalize()   # es. Timestamp('2023-12-13 00:00:00')
        
        # 3.5.2) Build the 168-hour index from that date.
        hourly_idx = pd.date_range(start=start_ts, periods=168, freq="h")
             
        # 3.5.3) Build the hourly cooling series for the current season.
        cooling_column = find_compatible_column(
            cooling_week_df,
            season,
            COOLING_SEASON_COLUMN_ALIASES
        )
        s_hourly = pd.Series(
            cooling_week_df[cooling_column].values,
            index=hourly_idx,
            name="cooling_per_person_kW"
        )
        
        # 3.5.4) Reindex to all 10-minute timesteps defined in df_occ.
        #    method='ffill' spreads each hourly value over the following 6 timesteps.
        s_cool = s_hourly.reindex(
            index=df_occ.index,   # QUI il trucco: uso direttamente l’indice 10′ di df_occ
            method="ffill",
            limit=6          # no more than six 10-minute timesteps = 1 hour
        ) / 6                    # e divido for 6
        
        # 3.5.5) Fill any remaining NaN values with zero.
        s_cool = s_cool.fillna(0)
        
        # 3.5.6) Build the base DataFrame with the same shape as df_occ.
        df_cool_base = pd.DataFrame(        # opzione OTTIMIZZATA
            np.repeat(s_cool.values.astype(float)[:, None], df_occ.shape[1], axis=1),
            index=df_occ.index,
            columns=df_occ.columns,
            dtype=float
        )

        
        # 3.5.7) Apply the occupancy mask: away from home (0) -> zero, at home (1) -> s_cool.
        df_cooling_week = df_cool_base * df_occ    
        df_cooling_week.to_excel(cooling_writer, sheet_name=season)


        # 3.6) -------- Heating consumption --------
        # # Generate the seed and set the NumPy PRNG before the function call.
        # seed_for_heating = np.random.randint(0, 2**31 - 1)
        # np.random.seed(seed_for_heating)
        
        # Return kWh_el for heating.
        df_heat_week = calculate_heating_consumption_by_season(
            thermal_profile_cop_excel_path,
            annual_thermal_demand_th,
            season,
            season_periods,
            df_occ,
            min_absent_steps # number of consecutive timesteps with no people at home
        )
        # Save.
        df_heat_week.to_excel(heating_writer, sheet_name=season)


        # Replicate the same annual_thermal_demand generation to retrieve the exact values.
        # ----------------------------------------------------
        # Reset the same seed and regenerate the dictionary with the same logic.
        np.random.seed(seed_for_heating)
        annual_thermal_demand = {}
        for household_id, ftype in household_type_by_household.items():
            if ftype not in household_apartment_specs:
                raise KeyError(f"Household type '{ftype}' not found")
            _, a_min, a_max = household_apartment_specs[ftype]
            area = np.random.uniform(a_min, a_max)
            consumption_per_m2 = np.random.uniform(min_thermal_demand_per_m2, max_thermal_demand_per_m2)
            annual_thermal_demand[household_id] = area * consumption_per_m2  # kWh_th/year
    
        # annual_thermal_demand now contains the exact values used by the internal function.


        # -----------------------------------
        # 4) Seasonal and annual sums
        # -----------------------------------





        # --- Build 10-minute household-level DataFrames for each contribution, aligned to household_index. ---
        household_dhw_df    = persons_df_to_household_df(dhw_week_df, household_index)
        household_cooking_df   = persons_df_to_household_df(induction_week_df, household_index)
        household_cooling_df   = persons_df_to_household_df(df_cooling_week, household_index)
        household_ev_df    = persons_df_to_household_df(ev_demand_week_df, household_index)           # WARNING: this used to rely on df_charge_EV_home, which represents greedy battery charging; here we need the electricity demand associated with actual EV use.
        
        # df_heat_week is already household-level if the function returns household_id columns.
        # If it is not ordered, realign it:
        household_heating_df = df_heat_week.reindex(columns=household_index, fill_value=0.0).astype(float)
        
        # Store household-level DHW profiles to inspect possible simultaneous showers within the same household.
        household_dhw_dfs_by_season[season] = household_dhw_df.copy()

        # --- Household-level time series by load type, columns Household_i_<load>. ---
        household_load_ts_df = pd.concat(
            [
                household_dhw_df.add_prefix('').rename(columns=lambda c: f"{c}_DHW"),
                household_cooking_df.rename(columns=lambda c: f"{c}_Cook"),
                household_cooling_df.rename(columns=lambda c: f"{c}_Cool"),
                household_heating_df.rename(columns=lambda c: f"{c}_Heat"),
                household_ev_df.rename(columns=lambda c: f"{c}_EV"),
            ],
            axis=1
        )

        # Sort columns first by household and then by load type.
        # Example order: Household_1_DHW, Household_1_Cook, ..., Household_2_DHW, ...
        load_order = ["DHW","Cook","Cool","Heat","EV"]
        def sort_key(col):
            m = re.match(r'^(Household_\d+)_(.*)$', col)
            if not m:
                return (col, 999)
            household_id, load = m.group(1), m.group(2)
            return (int(household_id.split("_")[1]), load_order.index(load) if load in load_order else 999)

        household_load_ts_df = household_load_ts_df.reindex(sorted(household_load_ts_df.columns, key=sort_key), axis=1)

        # Save. un sheet for season
        household_load_ts_df.to_excel(household_load_timeseries_writer, sheet_name=season)


        # Sum contributions using NumPy, faster.
        dfs = [household_dhw_df, household_cooking_df, household_cooling_df, household_ev_df, household_heating_df]
        arrs = [d.values if d.values.dtype == np.float64 else d.values.astype(float) for d in dfs]
        sum_arr = np.nansum(np.stack(arrs, axis=0), axis=0)
        total_household_df = pd.DataFrame(sum_arr, index=dfs[0].index, columns=dfs[0].columns)


        # # Mi assicuro che tutti i df siano float (versione for evitare FutureWARNING)
        # household_dhw_df  = household_dhw_df.astype(float)
        # household_cooking_df = household_cooking_df.astype(float)
        # household_cooling_df = household_cooling_df.astype(float)
        # household_ev_df  = household_ev_df.astype(float)
        # household_heating_df = household_heating_df.astype(float)



        # # Somma dei contributi elettrici for timestep e for household
        # # Scale and units must be consistent; here, all values are assumed to be in kWh at 10-minute resolution.
        # total_household_df = household_dhw_df.add(household_cooking_df, fill_value=0.0)
        # total_household_df = total_household_df.add(household_cooling_df, fill_value=0.0)
        # total_household_df = total_household_df.add(household_ev_df, fill_value=0.0)
        # # Heating: household_heating_df is in kWh_el, as returned by the function -> add it.
        # total_household_df = total_household_df.add(household_heating_df, fill_value=0.0)
        
        # Optional: rename columns without 'Member' for cleaner labels; household_index already contains 'Household_X'.
        # total_household_df.columns are household_index already.
        
        # Save. il sheet for la season (time series con colonne Household_1,...)
        total_household_df.to_excel(household_timeseries_writer, sheet_name=season)


        # -----------------------------------
        # 5) Seasonal and annual sums
        # -----------------------------------
        
        # 5.1) Person-level weekly sums.
        dhw_person_week = dhw_week_df.sum(axis=0)        # series index=person
        cook_person_week = induction_week_df.sum(axis=0)
        person_ev_home_charging_week = df_charge_home_week.sum(axis=0)
        km_person_week = df_km.sum(axis=0)

        # 5.2) Aggregate to household level by summing members.
        household_dhw_week = aggregate_person_to_household_sum(dhw_person_week)
        household_cooking_week = aggregate_person_to_household_sum(cook_person_week)
        household_ev_home_charging_week = aggregate_person_to_household_sum(person_ev_home_charging_week)
        household_km_week = aggregate_person_to_household_sum(km_person_week)
        
        # 5.3) Scale to 13 weeks, DHW, cooking, EV home charging, km.
        household_dhw_season = household_dhw_week * 13.0
        household_cooking_season = household_cooking_week * 13.0
        household_ev_home_charging_season = household_ev_home_charging_week * 13.0
        household_km_season = household_km_week * 13.0

        # 5.4) Annual heating by household_id, using the annual_thermal_demand_th dictionary generated above.
        # Build the household index, using annual_thermal_demand_th because it is always defined here.
        household_index = pd.Index(sorted(annual_thermal_demand_th.keys()))
        household_heating_annual = pd.Series(annual_thermal_demand_th).reindex(household_index).fillna(0.0)

        # 5.5) Annualized cooling: if the current season is summer, calculate and store annual_cooling_by_household.
        if season == 'summer':
            # Weekly per-person cooling sum for the summer week.
            cooling_person_week = df_cooling_week.sum(axis=0)
            household_cooling_week = aggregate_person_to_household_sum(cooling_person_week)
            
            # Multiply by 16 and realign indices to households.
            household_cooling_annual = (household_cooling_week * 16.0).reindex(household_index).fillna(0.0)
            
            # Save. in una variabile esterna for riutilizzo nelle altre seasons
            annual_cooling_by_household = household_cooling_annual
        else:
            # If the variable was already calculated in summer, reuse it; otherwise use a zero series.
            if ('annual_cooling_by_household' in locals()) and (annual_cooling_by_household is not None):
                household_cooling_annual = annual_cooling_by_household.reindex(household_index).fillna(0.0)
            else:
                household_cooling_annual = pd.Series(0.0, index=household_index)

        # 5.6) Build the final DataFrame for the current season, aligning indices to household_heating_annual.index.
        # household_index = household_heating_annual.index
        household_season_df = pd.DataFrame(index=household_index)
        household_season_df['DHW_kWh_el_season'] = household_dhw_season.reindex(household_index).fillna(0.0)
        household_season_df['Cooking_kWh_el_season'] = household_cooking_season.reindex(household_index).fillna(0.0)
        household_season_df['EV_home_charging_kWh_el_season'] = household_ev_home_charging_season.reindex(household_index).fillna(0.0)
        household_season_df['km_season'] = household_km_season.reindex(household_index).fillna(0.0)
        household_season_df['Heating_kWh_th_annual'] = household_heating_annual
        household_season_df['Cooling_kWh_el_annual'] = household_cooling_annual
        
        household_type_series = pd.Series(household_type_by_household).reindex(household_index).fillna('Unknown')
        # Insert Household type as the first column.
        household_season_df.insert(0, 'Household type', household_type_series.values)
                

        # Save only the required seasonal columns.
        season_columns = ['Household type', 'DHW_kWh_el_season', 'Cooking_kWh_el_season', 'EV_home_charging_kWh_el_season', 'km_season']
        household_season_reduced_df = household_season_df.reindex(columns=season_columns).fillna(0.0)
        household_season_reduced_df.to_excel(household_annual_writer, sheet_name=season)
        
        # Store the DataFrame for the annual summary, keeping the full version for possible debugging.
        dfs_by_season[season] = household_season_df.copy()


        # -----------------------------------
        # 6) Cumulative-consumption plots. WARNING: these plots are generated only if consumption is calculated.
        # -----------------------------------
        
        # 6.1) Build a single DataFrame with all timestep-level sums.
        contribs = {
            'DHW':    dhw_week_df.sum(axis=1),
            'Cooking': induction_week_df.sum(axis=1),
            'Cooling': df_cooling_week.sum(axis=1),
            'EV': df_charge_home_week.sum(axis=1),
            # 'EV_work': df_charge_work_week.sum(axis=1),
            'Heating': df_heat_week.sum(axis=1)
        }
        additional_loads_weekly_aggregated_ts_df = pd.DataFrame(contribs)
    
        # 6.2) Save to Excel.
        additional_loads_weekly_aggregated_ts_df.to_excel(additional_total_writer, sheet_name=season)
        
        
        # Threshold parameter: skip individual plots when the absolute series sum is <= tol.
        tol = 1e-9   


        # 6.2b) Build alternative DataFrame:
        #       conventional base load + additional loads
        conventional_plus_additional_weekly_ts_df = None

        if PLOT_STACKED_CONV_PLUS_ADDITIONAL and base_load_xls is not None:
            base_total = _read_base_load_total_for_season(
                base_load_xls=base_load_xls,
                season=season,
                target_index=additional_loads_weekly_aggregated_ts_df.index,
                sheet_kind=BASE_LOAD_SHEET_KIND,
            )

            if base_total is not None:
                conventional_plus_additional_weekly_ts_df = pd.concat(
                    [
                        base_total.rename(BASE_LOAD_LABEL),
                        additional_loads_weekly_aggregated_ts_df
                    ],
                    axis=1
                )

                # 6.2c) Combined daily activity + conventional/additional consumption plot.
                # This plot is optional and does not replace plot_stacked_consumption_for_paper.
                if (
                    RUN_DAILY_ACTIVITY_CONSUMPTION_PLOT
                    and conventional_plus_additional_weekly_ts_df is not None
                    and _season_selected_for_daily_plot(season, DAILY_ACTIVITY_CONSUMPTION_PLOT_SEASONS)
                ):
                    selected_daily_plot_date = DAILY_ACTIVITY_CONSUMPTION_DATE_BY_SEASON.get(
                        season,
                        None
                    )

                    daily_activity_consumption_title = (
                        f"Daily activity profiles and energy demand - {season_english_dict[season]}\n"
                        f"{num_households} households - Zone: {selected_region}"
                    )

                    daily_activity_consumption_base_name = (
                        f"{num_households}_households_{selected_region}_"
                        f"activity_and_consumption_{season}_"
                        f"day{DAILY_ACTIVITY_CONSUMPTION_DAY_INDEX + 1}"
                    )

                    plot_daily_activity_and_consumption(
                        activity_week_df=activity_df,
                        consumption_week_df=conventional_plus_additional_weekly_ts_df,
                        title=daily_activity_consumption_title,
                        file_name=f"{daily_activity_consumption_base_name}_two_column.png",
                        output_folder=additional_consumption_output_pictures_folder,
                        selected_date=selected_daily_plot_date,
                        day_index=DAILY_ACTIVITY_CONSUMPTION_DAY_INDEX,
                        activity_categories=aggregated_activities,
                        activity_code_to_label=activity_code_to_label,
                        activity_english_dict=activity_english_dict,
                        activity_color_map=activity_color_map,
                        consumption_column_order=STACKED_CONSUMPTION_ORDER,
                        consumption_color_map=STACKED_CONSUMPTION_COLOR_MAP,
                        expected_steps_per_day=144,
                        show_title=SHOW_DAILY_ACTIVITY_CONSUMPTION_TITLE,
                        fig_size=(7.20, 5.80),
                        font_size=9.0,
                        label_font_size=10.0,
                        tick_font_size=8.5,
                        activity_legend_font_size=6.2,
                        consumption_legend_font_size=8.0,
                        title_font_size=11.0,
                        save_pdf=SAVE_DAILY_ACTIVITY_CONSUMPTION_PDF
                    )





        
        # # 6.3) ---------------------- Draw individual additional-load plots, weekly profiles ------------------------------
        # for nome, series in additional_loads_weekly_aggregated_ts_df.items():
        #     total = series.abs().sum()
        #     if total <= tol:
        #         # skip plot singolo se contributo nullo
        #         # print(f"[SKIP] {name} — {season}: zero contribution (sum={total:.3e})")
        #         continue
            
        #     fig, ax = plt.subplots(figsize=(12, 6), dpi=200)     # maggiore dpi e figsize
        #     series.plot.area(
        #         ax = ax,
        #         title = f'{name} — {season_english_dict[season]}\n{num_households} households - Zone: {selected_region}',
        #         ylabel = 'kWh'
        #     )
        #     ax.set_xlabel('')
        #     ax.grid(axis='y', linestyle=':', alpha=0.3)
        #     fig.tight_layout()
        #     outpath = os.path.join(
        #         additional_consumption_output_pictures_folder,
        #         f'{num_households}_households_{selected_region}_plot_{nome}_{season}.png'
        #     )
        #     fig.savefig(outpath, dpi=200, bbox_inches='tight', pad_inches=0.03)
        #     plt.close(fig)
        # #-----------------------------------------------------------------------------------------------------------------------
    
        # 6.4) Stacked-area plot, all weekly additional loads.
        fig, ax = plt.subplots(figsize=(14, 7), dpi=200)
        additional_loads_weekly_aggregated_ts_df.plot.area(
            ax = ax,
            title = f'Stacked consumption — {season_english_dict[season]}\n{num_households} households - Zone: {selected_region}',
            ylabel = 'kWh',
            legend = True
        )
        ax.set_xlabel('')
        # Optional customization.
        ax.grid(axis='y', linestyle=':', alpha=0.3)
        fig.tight_layout()
        outpath = os.path.join(
            additional_consumption_output_pictures_folder,
            f'{num_households}_households_{selected_region}_plot_impilato_{season}.png'
        )
        fig.savefig(outpath, dpi=200, bbox_inches='tight', pad_inches=0.03)
        plt.close(fig)


        # 6.4b) Stacked-area plot, base load + additional loads:
        #       conventional/base load + additional loads
        if conventional_plus_additional_weekly_ts_df is not None:

            stacked_conventional_additional_title = (
                f"Stacked consumption (conv+additional) — {season_english_dict[season]}\n"
                f"{num_households} households - Zone: {selected_region}"
            )

            stacked_conventional_additional_base_name = (
                f"{num_households}_households_{selected_region}_"
                f"plot_impilato_conv_addiz_{season}"
            )

            # # ------------------------------------------------------------------
            # # (UNCOMMENT if to print a single-column sized graph)
            # # ------------------------------------------------------------------
            # plot_stacked_consumption_for_paper(
            #     df_plot=conventional_plus_additional_weekly_ts_df,
            #     title=stacked_conventional_additional_title,
            #     file_name=f"{stacked_conventional_additional_base_name}_single_column.png",
            #     output_folder=additional_consumption_output_pictures_folder,
            #     column_order=STACKED_CONSUMPTION_ORDER,
            #     color_map=STACKED_CONSUMPTION_COLOR_MAP,
            #     show_title=SHOW_STACKED_CONSUMPTION_TITLE,
            #     fig_size=(3.55, 2.80),
            #     font_size=8.0,
            #     label_font_size=8.5,
            #     tick_font_size=7.2,
            #     legend_font_size=6.8,
            #     title_font_size=9.0,
            #     legend_position="bottom",
            #     legend_ncol=3,
            #     y_label="Energy demand [kWh]",
            #     save_pdf=True
            # )

            # ------------------------------------------------------------------
            # Two-column / full-width version
            # Recommended for papers if the legend should remain on the right and readability should be high.
            # ------------------------------------------------------------------
            plot_stacked_consumption_for_paper(
                df_plot=conventional_plus_additional_weekly_ts_df,
                title=stacked_conventional_additional_title,
                file_name=f"{stacked_conventional_additional_base_name}_two_column.png",
                output_folder=additional_consumption_output_pictures_folder,
                column_order=STACKED_CONSUMPTION_ORDER,
                color_map=STACKED_CONSUMPTION_COLOR_MAP,
                show_title=SHOW_STACKED_CONSUMPTION_TITLE,
                fig_size=(7.20, 3.60),
                font_size=9.5,
                label_font_size=10.5,
                tick_font_size=9.0,
                legend_font_size=8.0,
                title_font_size=11.0,
                legend_position="right",
                legend_ncol=1,
                y_label="Energy demand [kWh]",
                save_pdf=True
            )





  
    # -------------------------
    # 7b) Build the annual sheet
    # -------------------------
    
    # 1) Ensure that the household index is consistent.
    if 'annual_thermal_demand' in locals():
        household_index = pd.Index(sorted(annual_thermal_demand.keys()))
    else:
        # Fallback: take the index from one of the dfs_by_season DataFrames, if available.
        any_df = next(iter(dfs_by_season.values()), None)
        if any_df is not None:
            household_index = any_df.index
        else:
            household_index = pd.Index([])
    
    # 2) Seasonal-to-annual sum for the required columns.
    # Initialize the annual DataFrame.
    annual_df = pd.DataFrame(index=household_index)
    
    # Seasonal columns to sum.
    seasonal_cols_map = {
        'DHW_kWh_el_annual': 'DHW_kWh_el_season',
        'Cooking_kWh_el_annual': 'Cooking_kWh_el_season',
        'EV_home_charging_kWh_el_annual': 'EV_home_charging_kWh_el_season',
        'annual_km': 'km_season'
    }
    
    # Sum across seasons.
    for out_col, season_col in seasonal_cols_map.items():
        # Sum over all available seasons.
        sums = None
        for season_name, season_df in dfs_by_season.items():
            if season_col in season_df.columns:
                if sums is None:
                    sums = season_df[season_col].reindex(household_index).fillna(0.0).astype(float)
                else:
                    sums = sums.add(season_df[season_col].reindex(household_index).fillna(0.0).astype(float), fill_value=0.0)
        if sums is None:
            sums = pd.Series(0.0, index=household_index)
        annual_df[out_col] = sums
    
    # 3) Annual heating: take annual_thermal_demand, kWh_th/year.
    if 'annual_thermal_demand' in locals():
        heating_series = pd.Series(annual_thermal_demand).reindex(household_index).fillna(0.0)
    else:
        heating_series = pd.Series(0.0, index=household_index)
    annual_df['Heating_kWh_th_annual'] = heating_series
    
    # 4) Annual cooling: reuse annual_cooling_by_household if it was calculated in summer.
    if ('annual_cooling_by_household' in locals()) and (annual_cooling_by_household is not None):
        cooling_series = annual_cooling_by_household.reindex(household_index).fillna(0.0)
    else:
        cooling_series = pd.Series(0.0, index=household_index)
    annual_df['Cooling_kWh_el_annual'] = cooling_series
    
    # 5) Optional: convert household-type options and rename columns as needed.
    # The current names already follow the requested convention.
    
    # Insert Household type as the first DataFrame column, which becomes the second Excel column after the index.
    annual_household_type_series = pd.Series(household_type_by_household).reindex(household_index).fillna('Unknown')
    annual_df.insert(0, 'Household type', annual_household_type_series.values)
    
    
    # 6) Write the annual sheet.
    annual_df.to_excel(household_annual_writer, sheet_name='Annual')


    # -----------------------------------
    # 8) Close all writers.
    # -----------------------------------
    writer_occ.close()
    dhw_writer.close()
    induction_writer.close()
    ev_home_charging_greedy_writer.close()
    ev_demand_writer.close()


    ev_work_charging_writer.close()
    ev_soc_writer.close()
    ev_km_writer.close()

    cooling_writer.close()
    heating_writer.close()
    
    additional_total_writer.close()
    household_annual_writer.close()

    household_timeseries_writer.close()

    household_load_timeseries_writer.close()
    dhw_overlap_writer.close()
    cooking_overlap_writer.close()

    #%% # Calculate distance travelled over the full year.
    # Outside the seasonal loop, calculate annual km.
    annual_km = weekly_km * 13
    annual_km.name = 'annual_km'
    
    # Optional: keep only persons with annual km > 0.
    annual_km = annual_km[annual_km > 0]


    # Save
    annual_km.to_frame().to_excel(
        os.path.join(
            additional_consumption_output_xls_folder,
            f'{num_households}_households_{selected_region}_annual_km_EV.xlsx'),
        sheet_name='annual'
    )

[INFO][BASE LOAD] Aggregated file not found.
[INFO][BASE LOAD] Automatically creating: output\02_base_consumption_Xls\146_households_Cen_ts_kWh_base_total_only.xlsx
[INFO][BASE LOAD] Full source file: output\02_base_consumption_Xls\146_households_Cen_ts_kWh_aggregated_households_base.xlsx
[OK][BASE LOAD] Aggregated 'winter TOT' -> 'winter TOT' (1008 time steps)
[OK][BASE LOAD] Aggregated 'spring TOT' -> 'spring TOT' (1008 time steps)
[OK][BASE LOAD] Aggregated 'summer TOT' -> 'summer TOT' (1008 time steps)
[OK][BASE LOAD] Aggregated 'autumn TOT' -> 'autumn TOT' (1008 time steps)
[OK][BASE LOAD] Saved aggregated file: output\02_base_consumption_Xls\146_households_Cen_ts_kWh_base_total_only.xlsx
[INFO][BASE LOAD] Loaded base-consumption file: output\02_base_consumption_Xls\146_households_Cen_ts_kWh_base_total_only.xlsx


C:\Users\Francesco\AppData\Local\Temp\ipykernel_10880\4262186653.py:236: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  dhw_count_by_household = dhw_binary.groupby(household_of_person, axis=1).sum()                         # 3) Sum by household: number of members with DHW consumption at each timestep
C:\Users\Francesco\AppData\Local\Temp\ipykernel_10880\4262186653.py:236: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  dhw_count_by_household = dhw_binary.groupby(household_of_person, axis=1).sum()                         # 3) Sum by household: number of members with DHW consumption at each timestep
C:\Users\Francesco\AppData\Local\Temp\ipykernel_10880\4262186653.py:236: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  dhw_count_by_household = dhw_binary.groupby(household_of_person, axis=1).sum()         

## 3.8 Additional-load plots (to be updated)
Plots for BESS SOC percentiles and similar outputs should be generated downstream, after running the HAGRID / 100Case model.  
Here, plots are generated to show annual-consumption percentiles across households for each additional-load class, such as cooking, cooling, and heating.

In [64]:
if RUN_ADDITIONAL_LOAD_PLOTS: # ------- TO BE UPDATED ----------------
    
    # ---------------------------
    # Annual-consumption percentile plots for additional loads, both overall and by household type.
    # ---------------------------
    
    # Container for global statistics.
    stats_rows = []

    cmap = plt.get_cmap('Blues')                # Choose the colormap.
    # Pick four evenly spaced colours, darker for the last one.
    colors_from_cmap = [cmap(v) for v in np.linspace(0.15, 0.75, 4)]

    
    # --------------------- PART A: global statistics for all households ---
    for full_col, short in categories_map.items():
        if full_col not in annual_df.columns:
            print(f"[WARN] Column '{full_col}' not found in annual_df — skipping")
            continue

        series = pd.to_numeric(annual_df[full_col], errors='coerce').dropna()
        if series.empty:
            print(f"[INFO] '{full_col}' has no valid data — skipping")
            continue

        # calcoli statistici
        perc_vals = np.percentile(series.values, qs)
        mean_val = float(series.mean())
        median_value = float(series.median())
        std_val = float(series.std())
        count_val = int(series.size)
        min_val, max_val = float(series.min()), float(series.max())

        row = {
            'category': full_col,
            'short': short,
            'count': count_val,
            'mean': mean_val,
            'median': median_value,
            'std': std_val,
            'min': min_val,
            'max': max_val
        }
        for q, v in zip(qs, perc_vals):
            row[f'p{q}'] = float(v)
        stats_rows.append(row)
        
        # Percentile plot.
        
        # Calculate percentiles, perc_vals follows the order of qs.
        perc_vals = np.percentile(series.values, qs)
        
        # Transform percentiles into a map for quick access.
        p_map = {q: v for q, v in zip(qs, perc_vals)}
        
        # Plotting setup: one x position only, 0.
        x = np.array([0])
        width = 0.6
        
        fig, ax = plt.subplots(figsize=(5,7), dpi=150)
        # Optional constraint to force the y-axis to start at 0; may be problematic for very large values, e.g. km/year.
        # ax.set_ylim(bottom=0)
        
        # Draw overlapping percentile bands as vertical bars, bottom=low and height=high-low.
        for (low_q, high_q, _), color in zip(bands, colors_from_cmap):
            low = p_map[low_q]
            high = p_map[high_q]
            ax.bar(x, high-low, bottom=low, width=width, color=color, alpha=0.95, edgecolor='none')

        # Horizontal median and mean lines.
        ax.axhline(median_value, color='k', linestyle='--', linewidth=1.6, label=f'Median = {median_value:.2f}')
        ax.axhline(mean_val, color='r', linestyle='-.', linewidth=1.4, label=f'Mean = {mean_val:.2f}')
        
        
        # Use the short contribution name as the x-axis label.
        ax.set_xticks(x)
        ax.set_xticklabels([short], rotation=0)
        ax.set_ylabel(f'{short} [kWh/year]')
        ax.set_title(f'Annual percentiles — {short} — {num_households} households')
        ax.grid(axis='y', linestyle=':', alpha=0.35)
        
        # Text annotations for key percentiles, p10, p50, p90.
        for q in (10,50,90):
            val = p_map[q]
            ax.text(x[0] + width/2 + 0.05, val, f'p{q}={val:.1f}', va='center', fontsize=9)
        
        # Build legend proxies for the percentile bands, using the same colours as in the plot.
        legend_handles = [Patch(facecolor=c, label=f'{low}-{high}%')
                        for (low,high,_), c in zip(bands, colors_from_cmap)]
        legend_handles += [Line2D([0],[0], color='k', linestyle='--', label='Median'),
                        Line2D([0],[0], color='r', linestyle='-.', label='Mean')]
        
        ax.legend(handles=legend_handles, loc='best')
        fig.tight_layout()
        
        out_pct = os.path.join(additional_consumption_output_pictures_folder, f'{num_households}_households_{selected_region}_percentile_bands_{short}_annual_tot.png')
        fig.savefig(out_pct, dpi=200, bbox_inches='tight', pad_inches=0.03)
        plt.close(fig)    

        
    # --------------------- PART B: percentiles by household type, with band plots for each category ---
    if 'Household type' not in annual_df.columns:
        print("[WARN] 'Household type' not found in annual_df — skipping household-type plots")
    else:
        household_types = annual_df['Household type'].dropna().unique().tolist()
        household_types = sorted(household_types)  # Adjust the order here if needed.

        # Band parameters, from the widest to the innermost band.
        bands = [(10,90,0.15),(20,80,0.25),(30,70,0.35),(40,60,0.45)]
        for full_col, short in categories_map.items():
            if full_col not in annual_df.columns:
                continue

            # raccolgo percentili/mean/median for ciascun household_type
            qs = [10,20,30,40,50,60,70,80,90]
            p_vals = {q: [] for q in qs}
            means = []
            median_values = []
            counts = []

            for household_type in household_types:
                vals = pd.to_numeric(annual_df.loc[annual_df['Household type'] == household_type, full_col], errors='coerce').dropna()
                if vals.empty:
                    for q in qs:
                        p_vals[q].append(np.nan)
                    means.append(np.nan)
                    median_values.append(np.nan)
                    counts.append(0)
                else:
                    perc = np.percentile(vals.values, qs)
                    for q, v in zip(qs, perc):
                        p_vals[q].append(v)
                    means.append(float(vals.mean()))
                    median_values.append(float(vals.median()))
                    counts.append(len(vals))

            p_arr = {q: np.array(p_vals[q], dtype=np.float64) for q in qs}
            means = np.array(means, dtype=np.float64)
            median_values = np.array(median_values, dtype=np.float64)
            counts = np.array(counts, dtype=int)

            x = np.arange(len(household_types))
            width = 0.6
            fig, ax = plt.subplots(figsize=(max(10, len(household_types)*1.4), 6), dpi=150)
            
            # Optional constraint to force the y-axis to start at 0; may be problematic for very large values, e.g. km/year.
            # ax.set_ylim(bottom=0)

            # Draw percentile bands.
            for (low_q, high_q, _), color in zip(bands, colors_from_cmap):
                low = p_arr[low_q]
                high = p_arr[high_q]
                height = high - low
                bottom = low
                low_plot = np.nan_to_num(low, nan=0.0)
                height_plot = np.nan_to_num(height, nan=0.0)
                # Use a single colour for each band.
                ax.bar(x, height, bottom=bottom, width=width, alpha=0.95, color=color, align='center', edgecolor='none')

            # Median and mean.
            ax.scatter(x, p_arr[50], marker='o', color='k', zorder=10, label='Median (p50)')
            ax.scatter(x, means, marker='x', color='r', zorder=11, label='Mean')

            ax.set_xticks(x)
            ax.set_xticklabels(household_types, rotation=30, ha='right')
            ax.set_title(f'Percentiles for type — {short} — {num_households} households')
            ax.set_ylabel(f'{short} [kWh/year]')
            ax.set_xlabel('Type of household')
            ax.grid(axis='y', linestyle=':', alpha=0.35)

            # Build legend using the same colours.
            legend_handles = [Patch(facecolor=c, label=f'{low}-{high}%') for (low,high,_), c in zip(bands, colors_from_cmap)]
            legend_handles += [Line2D([0],[0], color='k', marker='o', linestyle='None', label='Median'),
                            Line2D([0],[0], color='r', marker='x', linestyle='None', label='Mean')]
            
            ax.legend(handles=legend_handles, loc='best')


            # Annotate counts below the x-axis labels.
            ylim0, ylim1 = ax.get_ylim()
            for xi, cnt in zip(x, counts):
                ax.text(xi, ylim0 - 0.04*(ylim1-ylim0), str(int(cnt)), ha='center', va='top', fontsize=8)

            fig.tight_layout()
            out_bytype = os.path.join(additional_consumption_output_pictures_folder, f'{num_households}_households_{selected_region}_percentiles_by_household_type_{short}.png')
            fig.savefig(out_bytype, dpi=200, bbox_inches='tight', pad_inches=0.03)
            plt.close(fig)


    # #----------------------------------------------
    # # 2) DHW-consumption time-series plots by household, used to inspect shower overlaps. 
    # #----------------------------------------------
    
    # for season, household_dhw_df in household_dhw_dfs_by_season.items():

    #     fig, ax = plt.subplots(figsize=(14, 7), dpi=200)

    #     household_dhw_df.plot(
    #         ax=ax,
    #         linewidth=0.7,
    #         legend=False
    #     )

    #     ax.set_title(
    #         f'DHW by household — {season_english_dict[season]}\n'
    #         f'{num_households} households - Zone: {selected_region}'
    #     )
    #     ax.set_ylabel('kWh_el for 10 min')
    #     ax.set_xlabel('')
    #     ax.grid(axis='y', linestyle=':', alpha=0.3)

    #     fig.tight_layout()
    #     outpath = os.path.join(
    #         additional_consumption_output_pictures_folder,
    #         f'{num_households}_households_{selected_region}_cons_DHW_per_household_{season}.png'
    #     )
    #     fig.savefig(outpath, dpi=200, bbox_inches='tight', pad_inches=0.03)
    #     plt.close(fig)

## 3.XXX Conclusion

In [65]:
#%% ################# CONCLUSION #######################################

# Get the end time.
end_time = datetime.datetime.now()
print(f"End time : {end_time}")

# Calculate execution time.
execution_time = end_time - start_time
print(f"Execution time : {execution_time}")

End time : 2026-07-10 10:58:28.607558
Execution time : 0:21:29.816901
